# Full CPU workflow: baseline + two GNN replacements

Run this notebook from top to bottom. It rebuilds all five baseline collections
using the main notebook's model selection/training recipe, then trains GNNs only
for `layout:xla:default` and `layout:xla:random`. It writes:

- `full_run/submission_baseline.csv`: rebuilt baseline for all five collections.
- `full_run/submission_hybrid.csv`: GNN predictions for the two XLA layout
  collections, with every other row copied exactly from that baseline.
- `full_run/validation_comparison.csv`: common-sample comparison against the
  fitted baseline members from this run.
- `full_run/models/ensemble_members_by_collection.joblib`: baseline models to keep.

The original top-17% fitted models are unavailable. Retraining may give different
results because the old runtime's package versions or settings are unknown. The
hybrid is an experimental candidate, not a guaranteed improvement. This notebook
does not upload anything to Kaggle automatically. Compare the validation results
and retain the baseline CSV as a control before making a submission.

Full baseline selection over five collections takes longer than the earlier
two-collection GNN experiments. Save/download the output bundle after completion.

This version is CPU/Colab disk-safe: sampled train/validation configuration rows are
stored in a bounded cache, static graph preprocessing is reused, one background worker
prefetches the next sampled file, and complete test graphs are streamed in batches without
leaving expanded copies on disk.


## 1. Data (skip download if this runtime already has it)

In [ ]:
from pathlib import Path
data_candidates = [Path("/content/predict-ai-model-runtime"), Path("/content/data"),
                   Path.cwd() / "data", Path.cwd().parent / "data", Path.cwd() / "predict-ai-model-runtime"]
if any((p / "npz_all/npz").exists() for p in data_candidates):
    print("Existing data found; skipping download.")
else:
    import os
    from google.colab import userdata
    import kagglehub
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    kagglehub.competition_download("predict-ai-model-runtime", output_dir="/content/predict-ai-model-runtime")


## 2. Baseline setup and data paths

In [ ]:
import importlib.util
import subprocess
import sys

def ensure_package(package_name, import_name=None):
    """Install a package only when it is missing from the current runtime."""
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        print(f'Installing {package_name}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])

ensure_package('numpy')
ensure_package('pandas')
ensure_package('scikit-learn', 'sklearn')
ensure_package('joblib')
ensure_package('networkx')
ensure_package('matplotlib')
ensure_package('tqdm')

# Optional boosted-tree libraries. If installation/import fails, their experiments are skipped.
try:
    ensure_package('xgboost')
    XGBOOST_AVAILABLE = True
except Exception as exc:
    print('xgboost unavailable:', exc)
    XGBOOST_AVAILABLE = False

try:
    ensure_package('lightgbm')
    LIGHTGBM_AVAILABLE = True
except Exception as exc:
    print('lightgbm unavailable:', exc)
    LIGHTGBM_AVAILABLE = False

from pathlib import Path
import gc
import os
import warnings

os.environ.setdefault('MPLCONFIGDIR', str(Path('/tmp') / 'matplotlib'))

import joblib
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

if XGBOOST_AVAILABLE:
    try:
        from xgboost import XGBRegressor
    except Exception as exc:
        print('Could not import XGBRegressor:', exc)
        XGBOOST_AVAILABLE = False

if LIGHTGBM_AVAILABLE:
    try:
        from lightgbm import LGBMRegressor
    except Exception as exc:
        print('Could not import LGBMRegressor:', exc)
        LIGHTGBM_AVAILABLE = False

print('XGBoost status:', 'enabled' if XGBOOST_AVAILABLE else 'skipped')
print('LightGBM status:', 'enabled' if LIGHTGBM_AVAILABLE else 'skipped')

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
FIGURE_DIR = Path('figures')
FIGURE_DIR.mkdir(exist_ok=True)

import json, hashlib, shutil, zipfile
from contextlib import contextmanager
from threadpoolctl import threadpool_limits
CPU_THREADS = min(4, os.cpu_count() or 1)
FULL_OUTPUT = Path.cwd() / "full_run"
FULL_OUTPUT.mkdir(exist_ok=True)
print("Output:", FULL_OUTPUT)


In [ ]:
def find_data_root():
    candidates = [
        Path.cwd() / 'data',
        Path.cwd().parent / 'data',
        Path('/content/data'),
        Path.cwd() / 'predict-ai-model-runtime',
        Path.cwd().parent / 'predict-ai-model-runtime',
        Path('/content/predict-ai-model-runtime'),
        Path.cwd(),
    ]
    for candidate in candidates:
        if (candidate / 'npz_all' / 'npz').exists():
            return candidate
    raise FileNotFoundError('Could not find npz_all/npz. Put the Kaggle data folder in data/ or update find_data_root().')


def find_sample_submission_path(data_root):
    candidates = [
        data_root / 'sample_submission_Eugene.csv',
        data_root / 'sample_submission.csv',
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return data_root / 'sample_submission.csv'


DATA_ROOT = find_data_root()
NPZ_ROOT = DATA_ROOT / 'npz_all' / 'npz'
SAMPLE_SUBMISSION_PATH = find_sample_submission_path(DATA_ROOT)

COLLECTIONS = {
    'tile:xla': NPZ_ROOT / 'tile' / 'xla',
    'layout:xla:default': NPZ_ROOT / 'layout' / 'xla' / 'default',
    'layout:xla:random': NPZ_ROOT / 'layout' / 'xla' / 'random',
    'layout:nlp:default': NPZ_ROOT / 'layout' / 'nlp' / 'default',
    'layout:nlp:random': NPZ_ROOT / 'layout' / 'nlp' / 'random',
}

print('DATA_ROOT:', DATA_ROOT)
print('NPZ_ROOT:', NPZ_ROOT)
print('sample submission path:', SAMPLE_SUBMISSION_PATH)
print('sample submission exists:', SAMPLE_SUBMISSION_PATH.exists())

if not SAMPLE_SUBMISSION_PATH.is_file():
    raise FileNotFoundError("The competition sample submission is required for exact IDs and row lengths")


In [ ]:
def split_files(collection_name, split):
    return sorted((COLLECTIONS[collection_name] / split).glob('*.npz'))

def infer_model_family(file_stem):
    """Infer a coarse graph/model family from a TpuGraphs file stem."""
    stem = str(file_stem).lower()
    known_families = ['resnet', 'bert', 'albert', 'inception', 'efficientnet', 'mlperf', 'transformer', 'retinanet', 'mask_rcnn', 'mnasnet', 'alexnet', 'openai', 'shapemask', 'magenta', 'brax', 'ncf', 'xception', 'electra', 'talking-heads', 'trax', 'unet', 'experts']
    for family in known_families:
        if stem.startswith(family) or family in stem:
            return family
    if len(stem) >= 24 and all((ch in '0123456789abcdef' for ch in stem[:24])):
        return 'hashed_graph_id'
    return stem.split('_')[0].split('-')[0].split('.')[0]

def get_num_configs(data):
    if 'config_feat' in data:
        return data['config_feat'].shape[0]
    if 'node_config_feat' in data:
        return data['node_config_feat'].shape[0]
    raise KeyError('Could not find config_feat or node_config_feat')

## 3. Original feature and training recipe

Feature formulas and baseline model selection are retained from the repository's
main-derived notebook at `b89bd62be29b593894c9fc3fa4b85992dc582448`.
Only data access is changed: configuration arrays are extracted once and mapped
from disk, avoiding repeated full decompression. No competitor code is used.


In [ ]:
import threading
from collections import OrderedDict
from concurrent.futures import ThreadPoolExecutor

CACHE_ROOT = Path.cwd() / ".gnn_cache"
SAMPLED_CACHE_DIR = CACHE_ROOT / "sampled"
SAMPLED_CACHE_MAX_BYTES = 1024 ** 3  # hard cap: 1 GiB
CACHE_FREE_SPACE_RESERVE_BYTES = 2 * 1024 ** 3  # keep 2 GiB free in Colab
CACHE_ROOT.mkdir(exist_ok=True)
SAMPLED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Remove only the legacy unbounded full-array cache created by older notebook versions.
legacy_cache_files = list(CACHE_ROOT.glob("*-node_config_feat.npy"))
for legacy_path in legacy_cache_files:
    legacy_path.unlink(missing_ok=True)
if legacy_cache_files:
    print(f"Removed {len(legacy_cache_files)} legacy full-array cache files")


def _read_npy_header(member):
    version = np.lib.format.read_magic(member)
    if version == (1, 0):
        return np.lib.format.read_array_header_1_0(member)
    if version in {(2, 0), (3, 0)}:
        return np.lib.format.read_array_header_2_0(member)
    raise ValueError(f"Unsupported .npy format version: {version}")


def npy_member_metadata(npz_path, array_name):
    """Read an embedded .npy header without expanding its payload."""
    member_name = array_name if array_name.endswith(".npy") else f"{array_name}.npy"
    with zipfile.ZipFile(npz_path) as archive:
        with archive.open(member_name) as member:
            shape, fortran_order, dtype = _read_npy_header(member)
    dtype = np.dtype(dtype)
    if dtype.hasobject:
        raise ValueError(f"Object arrays are not supported: {npz_path}:{array_name}")
    return tuple(shape), bool(fortran_order), dtype


def num_configs_in_npz(npz_path):
    with zipfile.ZipFile(npz_path) as archive:
        names = set(archive.namelist())
    array_name = "node_config_feat" if "node_config_feat.npy" in names else "config_feat"
    return int(npy_member_metadata(npz_path, array_name)[0][0])


def _read_exact(stream, size):
    chunks = []
    remaining = int(size)
    while remaining:
        chunk = stream.read(remaining)
        if not chunk:
            raise EOFError(f"Unexpected end of compressed array with {remaining} bytes missing")
        chunks.append(chunk)
        remaining -= len(chunk)
    return b"".join(chunks)


def read_npz_rows(npz_path, array_name, indices):
    """Stream selected C-order rows from a compressed .npy member in one forward pass."""
    indices = np.asarray(indices, dtype=np.int64)
    shape, fortran_order, dtype = npy_member_metadata(npz_path, array_name)
    if indices.ndim != 1 or len(np.unique(indices)) != len(indices):
        raise ValueError("Configuration indices must be one-dimensional and unique")
    if (indices < 0).any() or (indices >= shape[0]).any():
        raise IndexError(f"Configuration index outside [0, {shape[0]}): {npz_path}")
    if fortran_order:
        # TpuGraphs arrays are C-order. Keep a correct compatibility fallback for unusual files.
        with np.load(npz_path, allow_pickle=False) as archive:
            return np.asarray(archive[array_name][indices])
    result = np.empty((len(indices),) + shape[1:], dtype=dtype)
    if not len(indices):
        return result
    row_bytes = int(np.prod(shape[1:], dtype=np.int64)) * dtype.itemsize
    member_name = array_name if array_name.endswith(".npy") else f"{array_name}.npy"
    order = np.argsort(indices, kind="stable")
    with zipfile.ZipFile(npz_path) as archive:
        with archive.open(member_name) as member:
            _read_npy_header(member)
            payload_start = member.tell()
            for output_position in order:
                source_index = int(indices[output_position])
                member.seek(payload_start + source_index * row_bytes)
                raw = _read_exact(member, row_bytes)
                result[output_position] = np.frombuffer(raw, dtype=dtype).reshape(shape[1:])
    return result


def iter_npz_row_batches(npz_path, array_name, batch_size):
    """Yield every row exactly once while keeping a compressed member open."""
    shape, fortran_order, dtype = npy_member_metadata(npz_path, array_name)
    if fortran_order:
        with np.load(npz_path, allow_pickle=False) as archive:
            values = archive[array_name]
            for start in range(0, shape[0], batch_size):
                stop = min(start + batch_size, shape[0])
                yield np.arange(start, stop, dtype=np.int64), np.asarray(values[start:stop])
        return
    row_items = int(np.prod(shape[1:], dtype=np.int64))
    row_bytes = row_items * dtype.itemsize
    member_name = array_name if array_name.endswith(".npy") else f"{array_name}.npy"
    with zipfile.ZipFile(npz_path) as archive:
        with archive.open(member_name) as member:
            _read_npy_header(member)
            for start in range(0, shape[0], batch_size):
                stop = min(start + batch_size, shape[0])
                count = stop - start
                raw = _read_exact(member, count * row_bytes)
                rows = np.frombuffer(raw, dtype=dtype).copy().reshape((count,) + shape[1:])
                yield np.arange(start, stop, dtype=np.int64), rows


def load_npz_without_node_configs(npz_path):
    with np.load(npz_path, allow_pickle=False) as archive:
        return {name: archive[name] for name in archive.files if name != "node_config_feat"}


class SampledConfigCache:
    """Size-bounded disk cache containing only selected configuration rows."""
    def __init__(self, root=SAMPLED_CACHE_DIR, max_bytes=SAMPLED_CACHE_MAX_BYTES):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.max_bytes = int(max_bytes)
        self.lock = threading.Lock()
        self.hits = 0
        self.misses = 0
        self.memory_fallbacks = 0

    def path_for(self, npz_path, array_name, indices):
        path = Path(npz_path).resolve()
        stat = path.stat()
        indices = np.asarray(indices, dtype=np.int64)
        digest = hashlib.sha256(indices.tobytes()).hexdigest()[:16]
        token = f"{path}|{stat.st_size}|{stat.st_mtime_ns}|{array_name}|{digest}|{len(indices)}"
        key = hashlib.sha256(token.encode()).hexdigest()[:24]
        return self.root / f"sample-{key}.npy"

    def _files(self):
        return [path for path in self.root.glob("sample-*.npy") if path.is_file()]

    def _evict_for(self, required_bytes, protected=None):
        protected = Path(protected) if protected is not None else None
        files = self._files()
        total = sum(path.stat().st_size for path in files)
        for path in sorted(files, key=lambda item: item.stat().st_mtime_ns):
            if total + required_bytes <= self.max_bytes:
                break
            if protected is not None and path == protected:
                continue
            size = path.stat().st_size
            path.unlink(missing_ok=True)
            total -= size

    def materialize(self, npz_path, array_name, indices):
        indices = np.asarray(indices, dtype=np.int64)
        target = self.path_for(npz_path, array_name, indices)
        with self.lock:
            if target.exists():
                target.touch()
                self.hits += 1
                return np.load(target, mmap_mode="r", allow_pickle=False)
        rows = read_npz_rows(npz_path, array_name, indices)
        required = int(rows.nbytes + 4096)
        with self.lock:
            self._evict_for(required, protected=target)
            free_bytes = shutil.disk_usage(self.root).free
        if required > self.max_bytes or free_bytes < required + CACHE_FREE_SPACE_RESERVE_BYTES:
            self.memory_fallbacks += 1
            return rows
        with self.lock:
            if target.exists():
                target.touch()
                self.hits += 1
                return np.load(target, mmap_mode="r", allow_pickle=False)
            temporary = target.with_suffix(".tmp.npy")
            try:
                np.save(temporary, rows, allow_pickle=False)
                temporary.replace(target)
            finally:
                temporary.unlink(missing_ok=True)
            self.misses += 1
            return np.load(target, mmap_mode="r", allow_pickle=False)

    def remove(self, npz_path, array_name, indices):
        self.path_for(npz_path, array_name, indices).unlink(missing_ok=True)

    def disk_bytes(self):
        return sum(path.stat().st_size for path in self._files())

    def clear(self):
        for path in self._files():
            path.unlink(missing_ok=True)

    def report(self):
        return {"disk_mib": self.disk_bytes() / 1024**2, "hits": self.hits,
                "misses": self.misses, "memory_fallbacks": self.memory_fallbacks}


SAMPLED_CONFIG_CACHE = SampledConfigCache()


@contextmanager
def cached_npz(path, config_indices=None):
    """Load small members normally and only requested layout configurations."""
    path = Path(path).resolve()
    data = load_npz_without_node_configs(path)
    with zipfile.ZipFile(path) as archive:
        has_node_configs = "node_config_feat.npy" in archive.namelist()
    if has_node_configs:
        if config_indices is None:
            config_indices = np.arange(num_configs_in_npz(path), dtype=np.int64)
            data["node_config_feat"] = read_npz_rows(path, "node_config_feat", config_indices)
        else:
            data["node_config_feat"] = SAMPLED_CONFIG_CACHE.materialize(
                path, "node_config_feat", np.asarray(config_indices, dtype=np.int64)
            )
    try:
        yield data
    finally:
        data.clear()
        gc.collect()


def iter_config_batches(npz_path, batch_size=256):
    """Stream test configurations without writing expanded full arrays to disk."""
    static = load_npz_without_node_configs(npz_path)
    if "config_feat" in static:
        values = static["config_feat"]
        for start in range(0, len(values), batch_size):
            stop = min(start + batch_size, len(values))
            batch = dict(static)
            batch["config_feat"] = np.asarray(values[start:stop])
            yield np.arange(start, stop, dtype=np.int64), batch
    else:
        for source_indices, values in iter_npz_row_batches(npz_path, "node_config_feat", batch_size):
            batch = dict(static)
            batch["node_config_feat"] = values
            yield source_indices, batch


In [ ]:
import time
import zlib


FEATURE_HASH_BINS = 128
WL_DEPTH = 3


FEATURE_EXPERIMENTS = {
    'simple_summary_ablation': {
        'use_degree_features': False,
        'use_dag_depth_features': False,
        'use_opcode_transition_features': False,
        'use_repeated_subgraph_features': False,
        'use_wl_features': False,
        'use_layout_local_graph_features': False,
    },
    'paper_mlp_baseline': {
        'use_degree_features': False,
        'use_dag_depth_features': False,
        'use_opcode_transition_features': False,
        'use_repeated_subgraph_features': False,
        'use_wl_features': False,
        'use_layout_local_graph_features': False,
    },
    'repeated_subgraph': {
        'use_degree_features': True,
        'use_dag_depth_features': True,
        'use_opcode_transition_features': True,
        'use_repeated_subgraph_features': True,
        'use_wl_features': False,
        'use_layout_local_graph_features': True,
    },
    'wl_fingerprint': {
        'use_degree_features': True,
        'use_dag_depth_features': True,
        'use_opcode_transition_features': False,
        'use_repeated_subgraph_features': False,
        'use_wl_features': True,
        'use_layout_local_graph_features': True,
    },
    'combined_compact_graph': {
        'use_degree_features': True,
        'use_dag_depth_features': True,
        'use_opcode_transition_features': True,
        'use_repeated_subgraph_features': True,
        'use_wl_features': True,
        'use_layout_local_graph_features': True,
    },
}

DEFAULT_FEATURE_PROFILE_NAME = 'combined_compact_graph'
ACTIVE_FEATURE_SETTINGS = FEATURE_EXPERIMENTS[DEFAULT_FEATURE_PROFILE_NAME]


def merge_feature_settings(feature_settings=None):
    settings = FEATURE_EXPERIMENTS['simple_summary_ablation'].copy()
    if feature_settings is None:
        settings.update(ACTIVE_FEATURE_SETTINGS)
    else:
        settings.update(feature_settings)
    return settings


def stable_hash_to_bin(value, n_bins=FEATURE_HASH_BINS):
    """Deterministic hash binning for graph patterns."""
    if not isinstance(value, bytes):
        value = str(value).encode('utf-8')
    return zlib.crc32(value) % n_bins


def safe_numeric_stats(prefix, arr):
    """Small aggregate stats. These are cheap and work for arrays of different shapes."""
    arr = np.asarray(arr)
    values = arr[np.isfinite(arr)] if np.issubdtype(arr.dtype, np.number) else np.array([])
    if values.size == 0:
        return {
            f'{prefix}_mean': 0.0,
            f'{prefix}_std': 0.0,
            f'{prefix}_min': 0.0,
            f'{prefix}_max': 0.0,
        }
    return {
        f'{prefix}_mean': float(values.mean()),
        f'{prefix}_std': float(values.std()),
        f'{prefix}_min': float(values.min()),
        f'{prefix}_max': float(values.max()),
    }


def distribution_stats(prefix, values):
    """Fixed summary columns for one-dimensional graph statistics."""
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0:
        values = np.array([0.0])
    stats = safe_numeric_stats(prefix, values)
    for percentile in [10, 25, 50, 75, 90]:
        stats[f'{prefix}_p{percentile}'] = float(np.percentile(values, percentile))
    return stats


def build_adjacency(edge_index, node_count):
    """Return incoming and outgoing adjacency lists for a directed graph."""
    incoming = [[] for _ in range(node_count)]
    outgoing = [[] for _ in range(node_count)]
    for src, dst in np.asarray(edge_index, dtype=np.int64):
        if 0 <= src < node_count and 0 <= dst < node_count:
            outgoing[int(src)].append(int(dst))
            incoming[int(dst)].append(int(src))
    return incoming, outgoing


def degree_features(edge_index, node_count):
    """Degree summaries preserve more graph structure than edge count alone."""
    incoming, outgoing = build_adjacency(edge_index, node_count)
    in_degree = np.array([len(nodes) for nodes in incoming], dtype=np.float64)
    out_degree = np.array([len(nodes) for nodes in outgoing], dtype=np.float64)
    total_degree = in_degree + out_degree

    features = {}
    features.update(distribution_stats('in_degree', in_degree))
    features.update(distribution_stats('out_degree', out_degree))
    features.update(distribution_stats('total_degree', total_degree))
    features['source_node_frac'] = float(np.mean(in_degree == 0)) if node_count else 0.0
    features['sink_node_frac'] = float(np.mean(out_degree == 0)) if node_count else 0.0
    return features


def longest_dag_depths(edge_index, node_count, reverse=False):
    """Longest-path depth from sources. If cycles appear, unresolved nodes stay at zero."""
    if node_count == 0:
        return np.array([], dtype=np.float64)

    edges = np.asarray(edge_index, dtype=np.int64)
    if reverse:
        edges = edges[:, [1, 0]]

    incoming, outgoing = build_adjacency(edges, node_count)
    indegree = np.array([len(nodes) for nodes in incoming], dtype=np.int64)
    queue = [i for i, degree in enumerate(indegree) if degree == 0]
    depth = np.zeros(node_count, dtype=np.float64)
    head = 0

    while head < len(queue):
        node = queue[head]
        head += 1
        for nxt in outgoing[node]:
            if depth[nxt] < depth[node] + 1:
                depth[nxt] = depth[node] + 1
            indegree[nxt] -= 1
            if indegree[nxt] == 0:
                queue.append(nxt)

    return depth


def dag_depth_features(edge_index, node_count):
    """Summarize where nodes sit in the computation DAG."""
    source_depth = longest_dag_depths(edge_index, node_count, reverse=False)
    sink_depth = longest_dag_depths(edge_index, node_count, reverse=True)
    features = {}
    features.update(distribution_stats('source_depth', source_depth))
    features.update(distribution_stats('sink_depth', sink_depth))
    features['dag_longest_path_estimate'] = float(max(source_depth.max(initial=0.0), sink_depth.max(initial=0.0)))
    return features


def normalized_hash_counts(prefix, bin_ids, n_bins=FEATURE_HASH_BINS):
    counts = np.bincount(np.asarray(bin_ids, dtype=np.int64), minlength=n_bins)[:n_bins].astype(np.float64)
    total = counts.sum()
    if total > 0:
        counts /= total
    return {f'{prefix}_bin_{i}': float(value) for i, value in enumerate(counts)}


def opcode_transition_features(node_opcode, edge_index, n_bins=FEATURE_HASH_BINS):
    """Count directed opcode-to-opcode transitions along graph edges."""
    node_opcode = np.asarray(node_opcode, dtype=np.int64)
    bin_ids = []
    for src, dst in np.asarray(edge_index, dtype=np.int64):
        if 0 <= src < len(node_opcode) and 0 <= dst < len(node_opcode):
            pattern = f'{int(node_opcode[src])}>{int(node_opcode[dst])}'
            bin_ids.append(stable_hash_to_bin(pattern, n_bins))
    return normalized_hash_counts('opcode_transition', bin_ids, n_bins=n_bins)


def repeated_subgraph_features(node_opcode, edge_index, n_bins=FEATURE_HASH_BINS, max_neighbors_per_side=16):
    """Approximate repeated local subgraphs by hashing opcode neighborhoods."""
    node_opcode = np.asarray(node_opcode, dtype=np.int64)
    node_count = len(node_opcode)
    incoming, outgoing = build_adjacency(edge_index, node_count)
    bin_ids = []
    raw_patterns = []

    for node in range(node_count):
        in_ops = sorted(int(node_opcode[n]) for n in incoming[node])[:max_neighbors_per_side]
        out_ops = sorted(int(node_opcode[n]) for n in outgoing[node])[:max_neighbors_per_side]
        pattern = f'op={int(node_opcode[node])}|in={in_ops}|out={out_ops}'
        raw_patterns.append(pattern)
        bin_ids.append(stable_hash_to_bin(pattern, n_bins))

    features = normalized_hash_counts('repeat_subgraph', bin_ids, n_bins=n_bins)
    pattern_counts = pd.Series(raw_patterns).value_counts() if raw_patterns else pd.Series(dtype=np.int64)
    features['repeat_subgraph_unique_frac'] = float(len(pattern_counts) / max(node_count, 1))
    features['repeat_subgraph_max_frac'] = float(pattern_counts.iloc[0] / max(node_count, 1)) if len(pattern_counts) else 0.0
    features['repeat_subgraph_repeated_frac'] = float(np.mean(pattern_counts.to_numpy() > 1)) if len(pattern_counts) else 0.0
    return features


def wl_subtree_features(node_opcode, edge_index, depth=WL_DEPTH, n_bins=FEATURE_HASH_BINS):
    """Weisfeiler-Lehman subtree count features over opcode-labeled graph nodes."""
    node_opcode = np.asarray(node_opcode, dtype=np.int64)
    node_count = len(node_opcode)
    incoming, outgoing = build_adjacency(edge_index, node_count)
    neighbors = [sorted(set(incoming[i] + outgoing[i])) for i in range(node_count)]
    labels = [f'op_{int(op)}' for op in node_opcode]
    features = {}

    for round_id in range(depth + 1):
        bin_ids = [stable_hash_to_bin(label, n_bins) for label in labels]
        features.update(normalized_hash_counts(f'wl_round_{round_id}', bin_ids, n_bins=n_bins))
        if round_id == depth:
            break

        next_labels = []
        for node in range(node_count):
            neighbor_labels = sorted(labels[nbr] for nbr in neighbors[node])
            combined = labels[node] + '|' + '|'.join(neighbor_labels)
            next_labels.append(str(zlib.crc32(combined.encode('utf-8'))))
        labels = next_labels

    return features


def graph_level_features(data, feature_settings=None):
    """Features shared by every configuration inside the same graph file."""
    settings = merge_feature_settings(feature_settings)
    node_feat = data['node_feat']
    node_opcode = data['node_opcode']
    edge_index = data['edge_index']

    node_count = int(node_feat.shape[0])
    edge_count = int(edge_index.shape[0])

    features = {
        'node_count': node_count,
        'edge_count': edge_count,
        'edge_per_node': edge_count / max(node_count, 1),
        'opcode_unique': int(np.unique(node_opcode).size),
        'opcode_mean': float(np.mean(node_opcode)),
        'opcode_std': float(np.std(node_opcode)),
    }
    features.update(safe_numeric_stats('node_feat', node_feat))

    opcode_hist = np.bincount(node_opcode.astype(np.int64), minlength=128)[:128]
    opcode_hist = opcode_hist / max(opcode_hist.sum(), 1)
    for i, value in enumerate(opcode_hist):
        features[f'opcode_hist_{i}'] = float(value)

    if settings['use_degree_features']:
        features.update(degree_features(edge_index, node_count))
    if settings['use_dag_depth_features']:
        features.update(dag_depth_features(edge_index, node_count))
    if settings['use_opcode_transition_features']:
        features.update(opcode_transition_features(node_opcode, edge_index))
    if settings['use_repeated_subgraph_features']:
        features.update(repeated_subgraph_features(node_opcode, edge_index))
    if settings['use_wl_features']:
        features.update(wl_subtree_features(node_opcode, edge_index))

    return features


def choose_indices(n_items, max_items=None, seed=RANDOM_SEED):
    """Uniform fallback sampler for arrays without labels."""
    if max_items is None or n_items <= max_items:
        return np.arange(n_items)
    local_rng = np.random.default_rng(seed)
    return np.sort(local_rng.choice(n_items, size=max_items, replace=False))


def choose_runtime_stratified_indices(runtimes, max_items, seed=RANDOM_SEED):
    """Sample configs across runtime quantiles while always keeping fastest examples."""
    runtimes = np.asarray(runtimes, dtype=np.float64)
    valid_idx = np.flatnonzero(np.isfinite(runtimes) & (runtimes > 0))
    if max_items is None or len(valid_idx) <= max_items:
        return np.arange(len(runtimes))
    if len(valid_idx) == 0:
        return choose_indices(len(runtimes), max_items=max_items, seed=seed)

    local_rng = np.random.default_rng(seed)
    sorted_idx = valid_idx[np.argsort(runtimes[valid_idx])]
    fastest_count = max(1, int(max_items * 0.15))
    slowest_count = max(1, int(max_items * 0.05))
    selected = set(sorted_idx[:fastest_count].tolist())
    selected.update(sorted_idx[-slowest_count:].tolist())

    remaining = np.array([idx for idx in valid_idx if idx not in selected], dtype=np.int64)
    budget = max_items - len(selected)
    if budget > 0 and len(remaining) > 0:
        log_runtime = np.log1p(runtimes[remaining])
        ranked_remaining = remaining[np.argsort(log_runtime)]
        bins = np.array_split(ranked_remaining, min(10, len(ranked_remaining)))
        per_bin = max(1, budget // max(len(bins), 1))
        for bin_values in bins:
            if budget <= 0:
                break
            take = min(per_bin, len(bin_values), budget)
            chosen = local_rng.choice(bin_values, size=take, replace=False)
            selected.update(chosen.tolist())
            budget = max_items - len(selected)

    if len(selected) < max_items:
        remaining = np.array([idx for idx in valid_idx if idx not in selected], dtype=np.int64)
        if len(remaining) > 0:
            fill = local_rng.choice(remaining, size=min(max_items - len(selected), len(remaining)), replace=False)
            selected.update(fill.tolist())

    if len(selected) > max_items:
        selected = set(local_rng.choice(np.array(sorted(selected)), size=max_items, replace=False).tolist())
    return np.array(sorted(selected), dtype=np.int64)


def choose_config_indices(data, split, max_items=None, seed=RANDOM_SEED):
    """Choose configuration rows using labels when available and uniform sampling otherwise."""
    n_items = get_num_configs(data)
    if max_items is None or n_items <= max_items:
        return np.arange(n_items)
    if split in ['train', 'valid'] and 'config_runtime' in data:
        return choose_runtime_stratified_indices(data['config_runtime'], max_items=max_items, seed=seed)
    return choose_indices(n_items, max_items=max_items, seed=seed)


def masked_mean_and_std(mask, values):
    """Vectorized mean/std of node-level values over valid nodes for each config."""
    mask = np.asarray(mask, dtype=np.float64)
    values = np.asarray(values, dtype=np.float64)
    counts = np.maximum(mask.sum(axis=1), 1.0)
    mean = mask @ values / counts
    second = mask @ (values ** 2) / counts
    std = np.sqrt(np.maximum(second - mean ** 2, 0.0))
    return mean, std


def layout_config_local_graph_features(data, config_indices):
    """Graph-position summaries around layout-configurable nodes."""
    if 'node_config_feat' not in data or 'node_config_ids' not in data:
        return pd.DataFrame(index=np.arange(len(config_indices)))

    node_opcode = np.asarray(data['node_opcode'], dtype=np.float64)
    edge_index = data['edge_index']
    node_count = len(node_opcode)
    node_ids = np.asarray(data['node_config_ids'], dtype=np.int64)
    node_ids = np.clip(node_ids, 0, max(node_count - 1, 0))

    incoming, outgoing = build_adjacency(edge_index, node_count)
    in_degree = np.array([len(nodes) for nodes in incoming], dtype=np.float64)
    out_degree = np.array([len(nodes) for nodes in outgoing], dtype=np.float64)
    total_degree = in_degree + out_degree
    source_depth = longest_dag_depths(edge_index, node_count, reverse=False)
    sink_depth = longest_dag_depths(edge_index, node_count, reverse=True)

    selected = data['node_config_feat'][config_indices]
    valid_node_mask = np.any(selected != -1, axis=2)
    selected_no_pad = np.where(selected == -1, 0, selected)
    config_value_by_node = selected_no_pad.mean(axis=2)
    counts = np.maximum(valid_node_mask.sum(axis=1), 1)

    local_values = {
        'opcode': node_opcode[node_ids],
        'in_degree': in_degree[node_ids],
        'out_degree': out_degree[node_ids],
        'total_degree': total_degree[node_ids],
        'source_depth': source_depth[node_ids],
        'sink_depth': sink_depth[node_ids],
    }

    rows = {
        'layout_local_valid_node_frac': valid_node_mask.mean(axis=1),
        'layout_local_config_value_mean': config_value_by_node.sum(axis=1) / counts,
        'layout_local_config_value_std': np.sqrt(
            np.maximum(((config_value_by_node ** 2).sum(axis=1) / counts) - ((config_value_by_node.sum(axis=1) / counts) ** 2), 0.0)
        ),
    }

    mask_float = valid_node_mask.astype(np.float64)
    for name, values in local_values.items():
        mean, std = masked_mean_and_std(mask_float, values)
        rows[f'layout_local_{name}_mean'] = mean
        rows[f'layout_local_{name}_std'] = std
        rows[f'layout_local_config_x_{name}_mean'] = (config_value_by_node * values.reshape(1, -1)).sum(axis=1) / counts

    return pd.DataFrame(rows)


def config_features_from_file(data, collection_name, config_indices=None, feature_settings=None):
    """Return one DataFrame row per selected configuration."""
    settings = merge_feature_settings(feature_settings)

    if 'config_feat' in data:
        config_feat = data['config_feat']
        if config_indices is None:
            config_indices = np.arange(config_feat.shape[0])
        selected = config_feat[config_indices]
        rows = pd.DataFrame(selected, columns=[f'tile_config_feat_{i}' for i in range(selected.shape[1])])
        rows['config_feat_mean'] = selected.mean(axis=1)
        rows['config_feat_std'] = selected.std(axis=1)
        rows['config_feat_max'] = selected.max(axis=1)
        rows['config_feat_nonzero_frac'] = (selected != 0).mean(axis=1)
    else:
        node_config_feat = data['node_config_feat']
        if config_indices is None:
            config_indices = np.arange(node_config_feat.shape[0])
        selected = node_config_feat[config_indices]

        padding_frac = (selected == -1).mean(axis=(1, 2))
        selected_no_pad = np.where(selected == -1, 0, selected)

        pieces = []
        for stat_name, values in [
            ('mean', selected_no_pad.mean(axis=1)),
            ('std', selected_no_pad.std(axis=1)),
            ('min', selected_no_pad.min(axis=1)),
            ('max', selected_no_pad.max(axis=1)),
        ]:
            pieces.append(pd.DataFrame(values, columns=[f'layout_config_{stat_name}_{i}' for i in range(values.shape[1])]))
        rows = pd.concat(pieces, axis=1)
        rows['layout_config_padding_frac'] = padding_frac
        rows['num_configurable_nodes'] = node_config_feat.shape[1]

        if settings['use_layout_local_graph_features']:
            rows = pd.concat([rows, layout_config_local_graph_features(data, config_indices)], axis=1)

    rows['config_index'] = config_indices.astype(int)
    rows['is_tile_collection'] = int(collection_name.startswith('tile'))
    rows['is_layout_collection'] = int(collection_name.startswith('layout'))
    return rows


def feature_frame_from_data(data, npz_path, collection_name, split, local_indices,
                            source_indices, feature_settings=None, graph_features=None):
    if graph_features is None:
        graph_features = graph_level_features(data, feature_settings=feature_settings)
    config_df = config_features_from_file(
        data, collection_name, local_indices, feature_settings=feature_settings
    )
    config_df["config_index"] = np.asarray(source_indices, dtype=np.int64)
    for name, value in graph_features.items():
        config_df[name] = value
    config_df["collection"] = collection_name
    config_df["split"] = split
    config_df["file_stem"] = Path(npz_path).stem
    if split in ["train", "valid"]:
        config_df["runtime"] = np.asarray(data["config_runtime"])[source_indices].astype(np.float64)
    numeric_cols = config_df.select_dtypes(include=[np.number]).columns
    config_df[numeric_cols] = config_df[numeric_cols].replace([np.inf, -np.inf], 0).fillna(0)
    return config_df


def features_for_npz_file(npz_path, collection_name, split, max_configs=None, seed=RANDOM_SEED, feature_settings=None):
    """Build features after selecting IDs, so only sampled layout rows are cached."""
    metadata = load_npz_without_node_configs(npz_path)
    n_configs = num_configs_in_npz(npz_path)
    if split == "train" and "config_runtime" in metadata:
        source_indices = choose_runtime_stratified_indices(
            metadata["config_runtime"], max_configs, seed=seed
        )
    else:
        source_indices = choose_indices(n_configs, max_items=max_configs, seed=seed)
    with cached_npz(npz_path, config_indices=source_indices if "config_feat" not in metadata else None) as data:
        local_indices = (np.arange(len(source_indices), dtype=np.int64)
                         if "node_config_feat" in data else source_indices)
        return feature_frame_from_data(
            data, npz_path, collection_name, split, local_indices, source_indices,
            feature_settings=feature_settings,
        )



In [ ]:
QUICK_MODE = True
VALIDATION_PROFILE = 'medium'  # one of: 'quick', 'medium', 'final'


VALIDATION_PROFILES = {
    'quick': {
        'experiment_valid_files': 2,
        'experiment_valid_configs_per_file': 1000,
        'final_valid_files': 5,
        'final_valid_configs_per_file': 2000,
    },
    'medium': {
        'experiment_valid_files': 5,
        'experiment_valid_configs_per_file': 2500,
        'final_valid_files': 10,
        'final_valid_configs_per_file': 5000,
    },
    'final': {
        'experiment_valid_files': None,
        'experiment_valid_configs_per_file': 5000,
        'final_valid_files': None,
        'final_valid_configs_per_file': 10000,
    },
}

if VALIDATION_PROFILE not in VALIDATION_PROFILES:
    raise ValueError(f'Unknown VALIDATION_PROFILE: {VALIDATION_PROFILE}')

validation_profile = VALIDATION_PROFILES[VALIDATION_PROFILE]

if QUICK_MODE:
    MAX_TRAIN_FILES = {
        'tile:xla': 300,
        'layout:xla:default': 40,
        'layout:xla:random': 40,
        'layout:nlp:default': 60,
        'layout:nlp:random': 60,
    }
    MAX_TRAIN_CONFIGS_PER_FILE = {
        'tile:xla': 96,
        'layout:xla:default': 256,
        'layout:xla:random': 256,
        'layout:nlp:default': 256,
        'layout:nlp:random': 256,
    }
else:
    MAX_TRAIN_FILES = {name: None for name in COLLECTIONS}
    MAX_TRAIN_CONFIGS_PER_FILE = {name: None for name in COLLECTIONS}

MAX_VALID_FILES = validation_profile['final_valid_files']
MAX_VALID_CONFIGS_PER_FILE = validation_profile['final_valid_configs_per_file']

print('VALIDATION_PROFILE:', VALIDATION_PROFILE)
print('MAX_VALID_FILES:', MAX_VALID_FILES)
print('MAX_VALID_CONFIGS_PER_FILE:', MAX_VALID_CONFIGS_PER_FILE)


def select_files_for_split(collection_name, split, max_files=None, seed=RANDOM_SEED):
    """Select graph files, using graph-family stratification for training caps."""
    files = split_files(collection_name, split)
    if max_files is None or len(files) <= max_files:
        return files
    if split != 'train':
        return files[:max_files]

    grouped = {}
    for file_path in files:
        grouped.setdefault(infer_model_family(file_path.stem), []).append(file_path)

    local_rng = np.random.default_rng(seed)
    for family_files in grouped.values():
        local_rng.shuffle(family_files)

    selected = []
    families = sorted(grouped, key=lambda family: len(grouped[family]))
    while len(selected) < max_files and families:
        progressed = False
        for family in families:
            if grouped[family] and len(selected) < max_files:
                selected.append(grouped[family].pop(0))
                progressed = True
        if not progressed:
            break
    return sorted(selected)


def build_split_table(collection_name, split, max_files=None, max_configs_per_file=None, feature_settings=None):
    files = select_files_for_split(collection_name, split, max_files=max_files, seed=RANDOM_SEED)

    frames = []
    progress = tqdm(files, desc=f'{collection_name} {split} feature extraction', unit='file')
    for i, file_path in enumerate(progress):
        progress.set_postfix(file=file_path.stem[:18])
        frame = features_for_npz_file(
            file_path,
            collection_name=collection_name,
            split=split,
            max_configs=max_configs_per_file,
            seed=RANDOM_SEED + i,
            feature_settings=feature_settings,
        )
        frames.append(frame)

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def validate_feature_frame(frame, split):
    numeric_cols = frame.select_dtypes(include=[np.number]).columns
    numeric_values = frame[numeric_cols].to_numpy(dtype=np.float64, copy=False)
    assert np.isfinite(numeric_values).all(), 'Feature frame contains NaN or infinite values.'
    assert 'config_index' in frame.columns, 'config_index must be preserved for submission ranking.'
    if split in ['train', 'valid']:
        assert 'runtime' in frame.columns, 'Train/valid frames must include runtime labels.'
    return {
        'rows': len(frame),
        'columns': frame.shape[1],
        'numeric_columns': len(numeric_cols),
        'has_runtime': 'runtime' in frame.columns,
    }




In [ ]:
def tile_top5_score(y_true, y_pred):
    """Approximate the tile competition metric for one graph."""
    order = np.argsort(y_pred)
    best_true_runtime = np.min(y_true)
    best_runtime_in_predicted_top5 = np.min(y_true[order[:5]])
    return 2.0 - (best_runtime_in_predicted_top5 / best_true_runtime)


def sampled_kendall_score(y_true, y_pred, max_pairs=20000, seed=RANDOM_SEED):
    """Fast sampled Kendall-style score in [-1, 1]. Higher is better."""
    n = len(y_true)
    if n < 2:
        return np.nan
    local_rng = np.random.default_rng(seed)
    i = local_rng.integers(0, n, size=max_pairs)
    j = local_rng.integers(0, n, size=max_pairs)
    mask = i != j
    i, j = i[mask], j[mask]

    true_order = np.sign(y_true[i] - y_true[j])
    pred_order = np.sign(y_pred[i] - y_pred[j])
    useful = (true_order != 0) & (pred_order != 0)
    if useful.sum() == 0:
        return np.nan
    return float(np.mean(true_order[useful] == pred_order[useful]) * 2 - 1)


## 4. Fit and save all five baseline collections

This executes the main recipe's comparison and final refit: layout XLA final
models use up to 40 graphs and 256 configurations per graph, rather than the
smaller reference models from the preceding GNN experiment. Available XGBoost
and LightGBM models participate as in the main notebook. Package availability
is recorded; this cannot certify an exact reproduction of the lost run.


In [ ]:
with threadpool_limits(limits=CPU_THREADS):
    NON_FEATURE_COLUMNS = {'collection', 'split', 'file_stem', 'runtime', 'model_family'}
    
    
    RUN_EXPERIMENT_COMPARISON = True
    TARGET_MODE = 'graph_centered_log_runtime'
    USE_FAMILY_BALANCING = True
    USE_RANK_ENSEMBLE = True
    ENSEMBLE_TOP_K = 2
    ENSEMBLE_MAX_RANKING_SCORE_GAP = 0.03
    
    BASE_EXPERIMENT_NAMES = [
        'paper_mlp_baseline',
        'simple_summary_ablation',
        'repeated_subgraph_hgb',
        'wl_fingerprint_hgb',
        'combined_compact_graph_hgb',
    ]
    
    OPTIONAL_EXPERIMENT_NAMES = []
    if XGBOOST_AVAILABLE:
        OPTIONAL_EXPERIMENT_NAMES.extend([
            'simple_summary_xgb',
            'combined_compact_graph_xgb',
        ])
    if LIGHTGBM_AVAILABLE:
        OPTIONAL_EXPERIMENT_NAMES.extend([
            'simple_summary_lgbm',
            'combined_compact_graph_lgbm',
        ])
    
    EXPERIMENT_NAMES = BASE_EXPERIMENT_NAMES + OPTIONAL_EXPERIMENT_NAMES
    
    EXPERIMENT_MODEL_TYPES = {
        'paper_mlp_baseline': 'mlp',
        'simple_summary_ablation': 'hgb',
        'repeated_subgraph_hgb': 'hgb',
        'wl_fingerprint_hgb': 'hgb',
        'combined_compact_graph_hgb': 'hgb',
        'simple_summary_xgb': 'xgb',
        'combined_compact_graph_xgb': 'xgb',
        'simple_summary_lgbm': 'lgbm',
        'combined_compact_graph_lgbm': 'lgbm',
    }
    
    EXPERIMENT_FEATURE_SETTINGS = {
        'paper_mlp_baseline': FEATURE_EXPERIMENTS['paper_mlp_baseline'],
        'simple_summary_ablation': FEATURE_EXPERIMENTS['simple_summary_ablation'],
        'repeated_subgraph_hgb': FEATURE_EXPERIMENTS['repeated_subgraph'],
        'wl_fingerprint_hgb': FEATURE_EXPERIMENTS['wl_fingerprint'],
        'combined_compact_graph_hgb': FEATURE_EXPERIMENTS['combined_compact_graph'],
        'simple_summary_xgb': FEATURE_EXPERIMENTS['simple_summary_ablation'],
        'combined_compact_graph_xgb': FEATURE_EXPERIMENTS['combined_compact_graph'],
        'simple_summary_lgbm': FEATURE_EXPERIMENTS['simple_summary_ablation'],
        'combined_compact_graph_lgbm': FEATURE_EXPERIMENTS['combined_compact_graph'],
    }
    
    print('XGBOOST_AVAILABLE:', XGBOOST_AVAILABLE)
    print('LIGHTGBM_AVAILABLE:', LIGHTGBM_AVAILABLE)
    print('Base experiments:', BASE_EXPERIMENT_NAMES)
    print('Optional experiments enabled:', OPTIONAL_EXPERIMENT_NAMES if OPTIONAL_EXPERIMENT_NAMES else 'none')
    if not XGBOOST_AVAILABLE:
        print('Skipping XGBoost experiments: xgboost is unavailable in this runtime')
    if not LIGHTGBM_AVAILABLE:
        print('Skipping LightGBM experiments: lightgbm is unavailable in this runtime')
    print('Final experiment run order:', EXPERIMENT_NAMES)
    print('USE_RANK_ENSEMBLE:', USE_RANK_ENSEMBLE)
    print('ENSEMBLE_TOP_K:', ENSEMBLE_TOP_K)
    print('ENSEMBLE_MAX_RANKING_SCORE_GAP:', ENSEMBLE_MAX_RANKING_SCORE_GAP)
    
    # Used when RUN_EXPERIMENT_COMPARISON is False, or as a fallback if a validation run fails.
    # These defaults come from the first Colab experiment output and can be overwritten by the
    # automatic validation-based selector below.
    DEFAULT_FINAL_EXPERIMENT_BY_COLLECTION = {
        'tile:xla': 'paper_mlp_baseline',
        'layout:xla:default': 'wl_fingerprint_hgb',
        'layout:xla:random': 'wl_fingerprint_hgb',
        'layout:nlp:default': 'simple_summary_ablation',
        'layout:nlp:random': 'simple_summary_ablation',
    }
    
    EXPERIMENT_MAX_TRAIN_FILES = {
        'tile:xla': 80,
        'layout:xla:default': 12,
        'layout:xla:random': 12,
        'layout:nlp:default': 12,
        'layout:nlp:random': 12,
    }
    EXPERIMENT_MAX_TRAIN_CONFIGS_PER_FILE = {
        'tile:xla': 64,
        'layout:xla:default': 128,
        'layout:xla:random': 128,
        'layout:nlp:default': 128,
        'layout:nlp:random': 128,
    }
    EXPERIMENT_MAX_VALID_FILES = validation_profile['experiment_valid_files']
    EXPERIMENT_MAX_VALID_CONFIGS_PER_FILE = validation_profile['experiment_valid_configs_per_file']
    print('EXPERIMENT_MAX_VALID_FILES:', EXPERIMENT_MAX_VALID_FILES)
    print('EXPERIMENT_MAX_VALID_CONFIGS_PER_FILE:', EXPERIMENT_MAX_VALID_CONFIGS_PER_FILE)
    
    def add_model_family_column(frame):
        frame = frame.copy()
        frame['model_family'] = frame['file_stem'].map(infer_model_family)
        return frame
    
    
    def family_balance_weights(train_df):
        """Inverse-frequency row weights by graph family, normalized to mean 1."""
        if not USE_FAMILY_BALANCING:
            return None
        family_counts = train_df['model_family'].value_counts()
        weights = train_df['model_family'].map(lambda family: 1.0 / family_counts[family]).to_numpy(dtype=np.float64).copy()
        weights *= len(weights) / max(weights.sum(), 1e-12)
        return weights
    
    
    def fit_model_with_optional_weights(model, X_train, y_train, sample_weight):
        if sample_weight is None:
            model.fit(X_train, y_train)
            return model
    
        try:
            model.fit(X_train, y_train, sample_weight=sample_weight)
        except (TypeError, ValueError):
            # Pipeline-based MLP baselines may not accept sample_weight directly.
            # Try routing it to the MLP step; if sklearn still rejects it, fall back
            # to unweighted fitting so the paper-style baseline remains runnable.
            try:
                model.fit(X_train, y_train, mlpregressor__sample_weight=sample_weight)
            except (TypeError, ValueError):
                model.fit(X_train, y_train)
        return model
    
    
    def make_runtime_model(model_type):
        if model_type == 'mlp':
            return make_pipeline(
                StandardScaler(),
                MLPRegressor(
                    hidden_layer_sizes=(128, 64),
                    activation='relu',
                    solver='adam',
                    alpha=1e-4,
                    batch_size=256,
                    learning_rate_init=1e-3,
                    max_iter=120,
                    early_stopping=True,
                    validation_fraction=0.15,
                    n_iter_no_change=10,
                    random_state=RANDOM_SEED,
                ),
            )
    
        if model_type == 'hgb':
            return HistGradientBoostingRegressor(
                loss='squared_error',
                learning_rate=0.06,
                max_iter=250,
                max_leaf_nodes=31,
                l2_regularization=0.01,
                random_state=RANDOM_SEED,
            )
    
        if model_type == 'xgb':
            if not XGBOOST_AVAILABLE:
                raise ImportError('xgboost is not available in this runtime')
            return XGBRegressor(
                objective='reg:squarederror',
                n_estimators=450,
                learning_rate=0.045,
                max_depth=6,
                min_child_weight=5,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_lambda=2.0,
                reg_alpha=0.0,
                tree_method='hist',
                random_state=RANDOM_SEED,
                n_jobs=-1,
                verbosity=0,
            )
    
        if model_type == 'lgbm':
            if not LIGHTGBM_AVAILABLE:
                raise ImportError('lightgbm is not available in this runtime')
            return LGBMRegressor(
                objective='regression',
                n_estimators=500,
                learning_rate=0.04,
                num_leaves=63,
                max_depth=-1,
                min_child_samples=20,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_lambda=2.0,
                random_state=RANDOM_SEED,
                n_jobs=-1,
                verbose=-1,
            )
    
        raise ValueError(f'Unknown model_type: {model_type}')
    
    
    def make_training_target(train_df, target_mode=TARGET_MODE):
        log_runtime = np.log1p(train_df['runtime'].to_numpy(dtype=np.float64))
        if target_mode == 'log_runtime':
            return log_runtime
        if target_mode == 'graph_centered_log_runtime':
            graph_median = train_df.assign(_log_runtime=log_runtime).groupby('file_stem')['_log_runtime'].transform('median')
            return log_runtime - graph_median.to_numpy(dtype=np.float64)
        raise ValueError(f'Unknown target_mode: {target_mode}')
    
    
    def validation_targets_and_scores(valid_df, raw_pred, target_mode=TARGET_MODE):
        y_true_runtime = valid_df['runtime'].to_numpy(dtype=np.float64)
        if target_mode == 'log_runtime':
            pred_for_ranking = np.expm1(raw_pred)
            true_for_mae = np.log1p(y_true_runtime)
            pred_for_mae = raw_pred
        elif target_mode == 'graph_centered_log_runtime':
            pred_for_ranking = raw_pred
            true_log = np.log1p(y_true_runtime)
            true_center = valid_df.assign(_log_runtime=true_log).groupby('file_stem')['_log_runtime'].transform('median')
            true_for_mae = true_log - true_center.to_numpy(dtype=np.float64)
            pred_for_mae = raw_pred
        else:
            raise ValueError(f'Unknown target_mode: {target_mode}')
        return y_true_runtime, pred_for_ranking, true_for_mae, pred_for_mae
    
    
    def evaluate_valid_files(
        model,
        feature_columns,
        collection_name,
        max_files=MAX_VALID_FILES,
        max_configs_per_file=MAX_VALID_CONFIGS_PER_FILE,
        feature_settings=None,
        target_mode=TARGET_MODE,
    ):
        files = split_files(collection_name, 'valid')
        if max_files is not None:
            files = files[:max_files]
    
        scores = []
        maes = []
        progress = tqdm(files, desc=f'{collection_name} validation', unit='file')
        for i, file_path in enumerate(progress):
            progress.set_postfix(file=file_path.stem[:18])
            valid_df = features_for_npz_file(
                file_path,
                collection_name=collection_name,
                split='valid',
                max_configs=max_configs_per_file,
                seed=RANDOM_SEED + i,
                feature_settings=feature_settings,
            )
            X_valid = valid_df.reindex(columns=feature_columns, fill_value=0)
            raw_pred = model.predict(X_valid)
            y_true, y_pred_for_ranking, y_true_for_mae, y_pred_for_mae = validation_targets_and_scores(
                valid_df,
                raw_pred,
                target_mode=target_mode,
            )
    
            maes.append(mean_absolute_error(y_true_for_mae, y_pred_for_mae))
            if collection_name.startswith('tile'):
                scores.append(tile_top5_score(y_true, y_pred_for_ranking))
            else:
                scores.append(sampled_kendall_score(y_true, y_pred_for_ranking, seed=RANDOM_SEED + i))
    
        return {
            'collection': collection_name,
            'valid_files': len(files),
            'target_mode': target_mode,
            'log_runtime_mae': float(np.mean(maes)) if maes else np.nan,
            'ranking_score': float(np.nanmean(scores)) if scores else np.nan,
        }
    
    
    def train_collection_model(
        collection_name,
        feature_settings,
        model_type,
        max_train_files,
        max_train_configs_per_file,
        max_valid_files,
        max_valid_configs_per_file,
        target_mode=TARGET_MODE,
    ):
        start = time.perf_counter()
        train_df = build_split_table(
            collection_name,
            'train',
            max_files=max_train_files,
            max_configs_per_file=max_train_configs_per_file,
            feature_settings=feature_settings,
        )
    
        train_df = add_model_family_column(train_df)
        family_counts = train_df[['file_stem', 'model_family']].drop_duplicates()['model_family'].value_counts().to_dict()
        print('Graph families:', family_counts)
    
        feature_columns = [col for col in train_df.columns if col not in NON_FEATURE_COLUMNS]
        X_train = train_df[feature_columns]
        y_train = make_training_target(train_df, target_mode=target_mode)
        sample_weight = family_balance_weights(train_df)
    
        model = make_runtime_model(model_type)
        fit_start = time.perf_counter()
        print(f'Fitting {model_type} model for {collection_name}: rows={len(X_train)}, features={len(feature_columns)}')
        model = fit_model_with_optional_weights(model, X_train, y_train, sample_weight)
        print(f'Finished fitting {model_type} model for {collection_name} in {time.perf_counter() - fit_start:.2f}s')
    
        validation = evaluate_valid_files(
            model,
            feature_columns,
            collection_name,
            max_files=max_valid_files,
            max_configs_per_file=max_valid_configs_per_file,
            feature_settings=feature_settings,
            target_mode=target_mode,
        )
        validation['model_type'] = model_type
        validation['family_balancing'] = USE_FAMILY_BALANCING
        validation['family_count'] = train_df['model_family'].nunique()
        validation['feature_count'] = len(feature_columns)
        validation['train_rows'] = len(train_df)
        validation['train_seconds'] = round(time.perf_counter() - start, 2)
    
        del train_df, X_train, y_train
        gc.collect()
        return model, feature_columns, validation
    
    
    def choose_final_experiments(experiment_results_df):
        if experiment_results_df.empty:
            return DEFAULT_FINAL_EXPERIMENT_BY_COLLECTION.copy()
    
        selected = {}
        for collection_name in COLLECTIONS:
            candidates = experiment_results_df[experiment_results_df['collection'] == collection_name].copy()
            if candidates.empty:
                selected[collection_name] = DEFAULT_FINAL_EXPERIMENT_BY_COLLECTION[collection_name]
                continue
    
            # Ranking is the competition objective. Feature count breaks near-ties toward simpler models.
            candidates = candidates.sort_values(
                ['ranking_score', 'feature_count'],
                ascending=[False, True],
            )
            selected[collection_name] = candidates.iloc[0]['experiment']
        return selected
    
    
    def choose_final_ensemble_experiments(experiment_results_df, top_k=ENSEMBLE_TOP_K, max_score_gap=ENSEMBLE_MAX_RANKING_SCORE_GAP):
        """Choose top validated experiments per collection for rank averaging."""
        if not USE_RANK_ENSEMBLE or top_k <= 1 or experiment_results_df.empty:
            return {
                collection_name: [experiment_name]
                for collection_name, experiment_name in choose_final_experiments(experiment_results_df).items()
            }
    
        selected = {}
        fallback = DEFAULT_FINAL_EXPERIMENT_BY_COLLECTION.copy()
        for collection_name in COLLECTIONS:
            candidates = experiment_results_df[experiment_results_df['collection'] == collection_name].copy()
            candidates = candidates[np.isfinite(candidates['ranking_score'])]
            if candidates.empty:
                selected[collection_name] = [fallback[collection_name]]
                continue
    
            candidates = candidates.sort_values(
                ['ranking_score', 'feature_count'],
                ascending=[False, True],
            )
            best_score = float(candidates.iloc[0]['ranking_score'])
            if max_score_gap is not None:
                candidates = candidates[candidates['ranking_score'] >= best_score - max_score_gap]
    
            experiments = candidates['experiment'].drop_duplicates().head(top_k).tolist()
            selected[collection_name] = experiments or [fallback[collection_name]]
        return selected
    
    
    experiment_results = []
    
    if RUN_EXPERIMENT_COMPARISON:
        for experiment_name in tqdm(EXPERIMENT_NAMES, desc='Experiment comparison', unit='experiment'):
            print('\n' + '#' * 80)
            print('Experiment:', experiment_name)
            feature_settings = EXPERIMENT_FEATURE_SETTINGS[experiment_name]
            model_type = EXPERIMENT_MODEL_TYPES[experiment_name]
    
            for collection_name in tqdm(COLLECTIONS, desc=f'{experiment_name} collections', unit='collection', leave=False):
                print('\n' + '=' * 80)
                print('Training collection:', collection_name)
                _, _, validation = train_collection_model(
                    collection_name,
                    feature_settings=feature_settings,
                    model_type=model_type,
                    max_train_files=EXPERIMENT_MAX_TRAIN_FILES[collection_name],
                    max_train_configs_per_file=EXPERIMENT_MAX_TRAIN_CONFIGS_PER_FILE[collection_name],
                    max_valid_files=EXPERIMENT_MAX_VALID_FILES,
                    max_valid_configs_per_file=EXPERIMENT_MAX_VALID_CONFIGS_PER_FILE,
                    target_mode=TARGET_MODE,
                )
                validation['experiment'] = experiment_name
                validation['baseline_reference'] = experiment_name == 'paper_mlp_baseline'
                experiment_results.append(validation)
                display(pd.DataFrame(experiment_results))
    
    experiment_results_df = pd.DataFrame(experiment_results)
    if not experiment_results_df.empty:
        summary_columns = [
            'experiment',
            'baseline_reference',
            'model_type',
            'collection',
            'target_mode',
            'family_balancing',
            'family_count',
            'ranking_score',
            'log_runtime_mae',
            'feature_count',
            'train_rows',
            'train_seconds',
        ]
        display(experiment_results_df[summary_columns].sort_values(['collection', 'experiment']))
    
        baseline_scores = (
            experiment_results_df[experiment_results_df['experiment'] == 'paper_mlp_baseline']
            [['collection', 'ranking_score', 'log_runtime_mae']]
            .rename(columns={
                'ranking_score': 'paper_mlp_ranking_score',
                'log_runtime_mae': 'paper_mlp_log_runtime_mae',
            })
        )
        comparison_to_paper_mlp = experiment_results_df.merge(baseline_scores, on='collection', how='left')
        comparison_to_paper_mlp['ranking_score_delta_vs_paper_mlp'] = (
            comparison_to_paper_mlp['ranking_score'] - comparison_to_paper_mlp['paper_mlp_ranking_score']
        )
        comparison_to_paper_mlp['log_runtime_mae_delta_vs_paper_mlp'] = (
            comparison_to_paper_mlp['log_runtime_mae'] - comparison_to_paper_mlp['paper_mlp_log_runtime_mae']
        )
        display(
            comparison_to_paper_mlp[
                [
                    'experiment',
                    'collection',
                    'ranking_score_delta_vs_paper_mlp',
                    'log_runtime_mae_delta_vs_paper_mlp',
                ]
            ].sort_values(['collection', 'experiment'])
        )
    
    
    # Train the final submission models. Each collection can use one winner or a small rank ensemble.
    final_experiment_by_collection = choose_final_experiments(experiment_results_df)
    final_ensemble_experiments_by_collection = choose_final_ensemble_experiments(experiment_results_df)
    
    display(
        pd.DataFrame(
            [
                {
                    'collection': collection_name,
                    'selected_final_experiment': final_experiment_by_collection[collection_name],
                    'rank_ensemble_experiments': ', '.join(final_ensemble_experiments_by_collection[collection_name]),
                    'ensemble_size': len(final_ensemble_experiments_by_collection[collection_name]),
                }
                for collection_name in COLLECTIONS
            ]
        )
    )
    
    models = {}
    feature_columns_by_collection = {}
    feature_settings_by_collection = {}
    model_type_by_collection = {}
    ensemble_members_by_collection = {}
    validation_rows = []
    
    print('\n' + '#' * 80)
    print('Training final per-collection selected models')
    print('Rank ensemble mode:', 'enabled' if USE_RANK_ENSEMBLE else 'disabled')
    
    for collection_name in tqdm(COLLECTIONS, desc='Final model training', unit='collection'):
        ensemble_experiments = final_ensemble_experiments_by_collection[collection_name]
        ensemble_members_by_collection[collection_name] = []
    
        print('\n' + '=' * 80)
        print('Training collection:', collection_name)
        print('Selected final experiment:', final_experiment_by_collection[collection_name])
        print('Rank ensemble experiments:', ensemble_experiments)
    
        for member_rank, final_experiment_name in enumerate(ensemble_experiments, start=1):
            feature_settings = EXPERIMENT_FEATURE_SETTINGS[final_experiment_name]
            model_type = EXPERIMENT_MODEL_TYPES[final_experiment_name]
    
            print('\n' + '-' * 80)
            print(f'Training ensemble member {member_rank}/{len(ensemble_experiments)}:', final_experiment_name)
    
            model, feature_columns, validation = train_collection_model(
                collection_name,
                feature_settings=feature_settings,
                model_type=model_type,
                max_train_files=MAX_TRAIN_FILES[collection_name],
                max_train_configs_per_file=MAX_TRAIN_CONFIGS_PER_FILE[collection_name],
                max_valid_files=MAX_VALID_FILES,
                max_valid_configs_per_file=MAX_VALID_CONFIGS_PER_FILE,
                target_mode=TARGET_MODE,
            )
    
            member = {
                'experiment': final_experiment_name,
                'model_type': model_type,
                'model': model,
                'feature_columns': feature_columns,
                'feature_settings': feature_settings,
            }
            ensemble_members_by_collection[collection_name].append(member)
    
            # Keep first member in the old dictionaries so diagnostics and saved-model code remain compatible.
            if member_rank == 1:
                models[collection_name] = model
                feature_columns_by_collection[collection_name] = feature_columns
                feature_settings_by_collection[collection_name] = feature_settings
                model_type_by_collection[collection_name] = model_type
    
            validation['experiment'] = final_experiment_name
            validation['ensemble_member_rank'] = member_rank
            validation['ensemble_size'] = len(ensemble_experiments)
            validation_rows.append(validation)
            display(pd.DataFrame(validation_rows))
    
    validation_df = pd.DataFrame(validation_rows)
    display(validation_df)


In [ ]:
MODEL_DIR = FULL_OUTPUT / 'models'
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(models, MODEL_DIR / 'baseline_models.joblib')
joblib.dump(feature_columns_by_collection, MODEL_DIR / 'baseline_feature_columns.joblib')
joblib.dump(feature_settings_by_collection, MODEL_DIR / 'feature_settings_by_collection.joblib')
joblib.dump(model_type_by_collection, MODEL_DIR / 'model_type_by_collection.joblib')
joblib.dump(final_experiment_by_collection, MODEL_DIR / 'final_experiment_by_collection.joblib')
joblib.dump(final_ensemble_experiments_by_collection, MODEL_DIR / 'final_ensemble_experiments_by_collection.joblib')
joblib.dump(ensemble_members_by_collection, MODEL_DIR / 'ensemble_members_by_collection.joblib')

print('Saved models to:', MODEL_DIR.resolve())

import importlib.metadata
versions = {}
for package in ["numpy", "pandas", "scikit-learn", "xgboost", "lightgbm", "torch"]:
    try: versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: versions[package] = None
(FULL_OUTPUT / "run_metadata.json").write_text(json.dumps({
    "source_commit": "b89bd62be29b593894c9fc3fa4b85992dc582448",
    "seed": RANDOM_SEED, "versions": versions, "experiments": EXPERIMENT_NAMES,
    "selected": final_ensemble_experiments_by_collection,
    "note": "Rebuilt baseline, not original top-17% fitted artifacts"}, indent=2))
experiment_results_df.to_csv(FULL_OUTPUT / "baseline_experiments.csv", index=False)


## 5. Generate the baseline submission

In [ ]:
def top_config_count(top_configs):
    return len([value for value in str(top_configs).split(';') if value != ''])

# Larger inference batches reduce pandas/model-call overhead while keeping
# layout configuration arrays bounded in memory.
BASELINE_PREDICT_BATCH_SIZE = 1024

def fit_ranking_to_expected_length(ordered_indices, expected_count, n_configs, collection_name):
    ordered_indices = [int(index) for index in ordered_indices]
    if expected_count is None:
        expected_count = 5 if collection_name.startswith('tile') else n_configs
    ranked = ordered_indices[:min(expected_count, n_configs)]
    if expected_count <= n_configs:
        return ranked
    if not collection_name.startswith('tile'):
        raise ValueError(
            f'Template requests {expected_count} configurations but only {n_configs} exist '
            f'for non-tile collection {collection_name}'
        )
    # The official tile template has five slots even when a local test graph
    # exposes fewer candidates. Preserve all valid ranked indices first, then
    # append unique placeholder positions to retain the required CSV shape.
    placeholders = list(range(n_configs, n_configs + expected_count - len(ranked)))
    return ranked + placeholders

def predict_ranking_for_file(file_path, collection_name, expected_count=None):
    members = ensemble_members_by_collection.get(collection_name)
    if not members:
        members = [{'experiment': final_experiment_by_collection[collection_name],
                    'model_type': model_type_by_collection[collection_name],
                    'model': models[collection_name],
                    'feature_columns': feature_columns_by_collection[collection_name],
                    'feature_settings': feature_settings_by_collection[collection_name]}]
    n_configs = num_configs_in_npz(file_path)
    if n_configs < 1:
        raise ValueError(f'No configurations found: {file_path}')
    member_scores = [np.empty(n_configs, dtype=np.float64) for _ in members]
    members_by_settings = OrderedDict()
    for member_id, member in enumerate(members):
        settings_key = json.dumps(member['feature_settings'], sort_keys=True)
        members_by_settings.setdefault(settings_key, []).append((member_id, member))
    graph_features = {}
    # Tile features are compact, so process each tile file in one vectorized call.
    # Layout arrays remain streamed to preserve the low-memory/low-disk behavior.
    batch_size = n_configs if collection_name.startswith('tile') else BASELINE_PREDICT_BATCH_SIZE
    for source_indices, data in iter_config_batches(file_path, batch_size=batch_size):
        local_indices = np.arange(len(source_indices), dtype=np.int64)
        for settings_key, grouped_members in members_by_settings.items():
            settings = grouped_members[0][1]['feature_settings']
            if settings_key not in graph_features:
                graph_features[settings_key] = graph_level_features(
                    data, feature_settings=settings
                )
            frame = feature_frame_from_data(
                data, file_path, collection_name, 'test', local_indices, source_indices,
                feature_settings=settings,
                graph_features=graph_features[settings_key],
            )
            for member_id, member in grouped_members:
                prediction = np.asarray(
                    member['model'].predict(frame.reindex(columns=member['feature_columns'], fill_value=0)),
                    dtype=np.float64,
                )
                member_scores[member_id][source_indices] = prediction
    rank_matrix = np.column_stack([
        pd.Series(scores).rank(method='average', ascending=True).to_numpy(dtype=np.float64)
        for scores in member_scores
    ])
    average_rank = pd.Series(rank_matrix.mean(axis=1), index=np.arange(n_configs)).sort_values(kind='mergesort')
    ordered_indices = fit_ranking_to_expected_length(
        average_rank.index.to_numpy(dtype=np.int64), expected_count=expected_count,
        n_configs=n_configs, collection_name=collection_name,
    )
    return ';'.join((str(int(i)) for i in ordered_indices))

def id_to_collection_and_stem(row_id):
    parts = row_id.split(':')
    return (':'.join(parts[:-1]), parts[-1])

def create_submission(output_path='submission.csv'):
    expected_counts_by_id = {}
    if SAMPLE_SUBMISSION_PATH.exists():
        template = pd.read_csv(SAMPLE_SUBMISSION_PATH)
        ids = template['ID'].tolist()
        expected_counts_by_id = dict(zip(template['ID'], template['TopConfigs'].map(top_config_count)))
    else:
        ids = []
        for collection_name in COLLECTIONS:
            for file_path in split_files(collection_name, 'test'):
                ids.append(f'{collection_name}:{file_path.stem}')
    rows = []
    progress = tqdm(ids, desc='Submission prediction', unit='row')
    for row_number, row_id in enumerate(progress, start=1):
        collection_name, file_stem = id_to_collection_and_stem(row_id)
        file_path = COLLECTIONS[collection_name] / 'test' / f'{file_stem}.npz'
        progress.set_postfix(collection=collection_name, ensemble_size=len(ensemble_members_by_collection.get(collection_name, [])))
        top_configs = predict_ranking_for_file(file_path, collection_name, expected_count=expected_counts_by_id.get(row_id))
        rows.append({'ID': row_id, 'TopConfigs': top_configs})
    submission = pd.DataFrame(rows)
    submission.to_csv(output_path, index=False)
    return submission

def validate_submission_frame(submission):
    required_columns = ['ID', 'TopConfigs']
    assert list(submission.columns) == required_columns, f'submission columns must be {required_columns}'
    assert not submission['ID'].duplicated().any(), 'submission contains duplicate IDs'
    assert submission['TopConfigs'].notna().all(), 'submission contains missing TopConfigs'
    expected_counts_by_id = {}
    if SAMPLE_SUBMISSION_PATH.exists():
        template = pd.read_csv(SAMPLE_SUBMISSION_PATH)
        assert submission['ID'].tolist() == template['ID'].tolist(), 'submission IDs/order do not match sample_submission.csv'
        expected_counts_by_id = dict(zip(template['ID'], template['TopConfigs'].map(top_config_count)))
    for row_id, top_configs in zip(submission['ID'], submission['TopConfigs']):
        collection_name, file_stem = id_to_collection_and_stem(row_id)
        assert collection_name in COLLECTIONS, f'unknown collection in ID: {row_id}'
        file_path = COLLECTIONS[collection_name] / 'test' / f'{file_stem}.npz'
        assert file_path.exists(), f'missing test npz for ID: {row_id}'
        values = [int(value) for value in str(top_configs).split(';') if value != '']
        n_configs = num_configs_in_npz(file_path)
        expected_count = expected_counts_by_id.get(row_id)
        if expected_count is None:
            expected_count = 5 if collection_name.startswith('tile') else n_configs
        assert values, f'empty TopConfigs for ID: {row_id}'
        assert len(values) == expected_count, f'TopConfigs length mismatch for ID: {row_id}'
        assert len(values) == len(set(values)), f'duplicate config index in TopConfigs for ID: {row_id}'
        valid_values = [value for value in values if 0 <= value < n_configs]
        if collection_name.startswith('tile'):
            assert len(valid_values) == min(expected_count, n_configs), f'tile row does not include enough valid configs for ID: {row_id}'
        else:
            assert len(valid_values) == expected_count, f'layout config index out of range for ID: {row_id}'
            if expected_count == n_configs:
                assert sorted(values) == list(range(n_configs)), f'layout row is not a full permutation for ID: {row_id}'
    return {'rows': len(submission), 'columns': list(submission.columns), 'unique_ids': int(submission['ID'].nunique()), 'sample_submission_match': bool(SAMPLE_SUBMISSION_PATH.exists())}

with threadpool_limits(limits=CPU_THREADS):
    baseline_submission = create_submission(FULL_OUTPUT / "submission_baseline.csv")
print(validate_submission_frame(baseline_submission))
print("Sampled cache after baseline:", SAMPLED_CONFIG_CACHE.report())


## 6. Train the two CPU GNNs and compare with this run's baseline

The previous observed recipes are used: 24 family-stratified training graphs for
`default`; the original first eight training graphs for `random`, which achieved
the higher diagnostic score in your earlier experiment. Both run for up to 20
epochs, validate every epoch, and restore the best checkpoint. These choices
were informed by the same five validation graphs, so results need confirmation
on broader held-out data. No per-file model switching is fitted to these scores.


In [ ]:
GNN_SOURCE = 'import importlib.util\nimport subprocess\nimport sys\nfrom pathlib import Path\nimport os\nimport time\nimport gc\nimport warnings\nimport json\nimport hashlib\nimport shutil\nimport zipfile\nfrom collections import OrderedDict\nfrom concurrent.futures import ThreadPoolExecutor\n\n\ndef ensure_package(package_name, import_name=None):\n    import_name = import_name or package_name\n    if importlib.util.find_spec(import_name) is None:\n        print(f"Installing {package_name}...")\n        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])\n\n\nensure_package("numpy")\nensure_package("pandas")\nensure_package("scikit-learn", "sklearn")\nensure_package("tqdm")\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import mean_absolute_error\nfrom tqdm.auto import tqdm\n\ntry:\n    import torch\n    import torch.nn as nn\n    import torch.nn.functional as F\n    TORCH_AVAILABLE = True\nexcept Exception as exc:\n    TORCH_AVAILABLE = False\n    raise RuntimeError("PyTorch is required for this GNN experiment notebook") from exc\n\nwarnings.filterwarnings("ignore")\nRANDOM_SEED = 42\nrng = np.random.default_rng(RANDOM_SEED)\ntorch.manual_seed(RANDOM_SEED)\n\nDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nprint("Torch:", torch.__version__)\nprint("Device:", DEVICE)\n\n# Keep these conservative for Colab RAM. Increase only after a successful smoke test.\nGNN_COLLECTIONS = ["layout:xla:default", "layout:xla:random"]\nGNN_SUBGRAPH_HOPS = 2\nGNN_MAX_SUBGRAPH_NODES = 512\nGNN_MAX_TRAIN_FILES = 24\nGNN_MAX_VALID_FILES = 5\nGNN_MAX_TRAIN_CONFIGS_PER_FILE = 64\nGNN_MAX_VALID_CONFIGS_PER_FILE = 1000\nGNN_BATCH_SIZE = 16\nGNN_PREDICT_BATCH_SIZE = 32\nGNN_HIDDEN_DIM = 32\nGNN_EPOCHS = 20\nGNN_LR = 1e-3\nPAIR_MARGIN = 0.1\nPAIR_MIN_LOG_RUNTIME_GAP = 0.02\nPAIR_MAX_PAIRS = 512\n\nprint("GNN_COLLECTIONS:", GNN_COLLECTIONS)\nprint("GNN_SUBGRAPH_HOPS:", GNN_SUBGRAPH_HOPS)\nprint("GNN_MAX_SUBGRAPH_NODES:", GNN_MAX_SUBGRAPH_NODES)\nprint("GNN_BATCH_SIZE:", GNN_BATCH_SIZE)\n\n# A single worker overlaps cache preparation; epoch training uses all capped CPU threads.\nGNN_PREFETCH_WORKERS = 1\nGNN_CPU_THREADS = min(4, os.cpu_count() or 1)\nif DEVICE.type == "cpu":\n    torch.set_num_threads(GNN_CPU_THREADS)\n\n# Sampled rows use the shared size-bounded cache. Full test arrays are streamed.\nGNN_CACHE_DIR = Path.cwd() / ".gnn_cache" / "gnn"\nGNN_GRAPH_CACHE_MB = 256\nGNN_OUTPUT_DIR = Path.cwd() / "gnn_runs"\nGNN_PATIENCE = 5\nGNN_MIN_DELTA = 1e-4\nGNN_VALIDATE_EVERY = 1\nprint("CPU threads:", torch.get_num_threads())\nprint("Training/inference batch sizes:", GNN_BATCH_SIZE, GNN_PREDICT_BATCH_SIZE)\nprint("Cache:", GNN_CACHE_DIR)\n\n# Optional: saved ensemble_members_by_collection.joblib from your main run.\n# Only load artifacts you created; joblib files can execute Python when loaded.\nBASELINE_ARTIFACT_PATH = None\nREFERENCE_FEATURE_PROFILES = ["simple_summary_ablation", "wl_fingerprint"]\nBASELINE_FEATURE_BATCH_SIZE = 64\nFIXED_VALIDATION = {}\n\n\ndef find_data_root():\n    candidates = [\n        Path.cwd() / "data",\n        Path.cwd().parent / "data",\n        Path("/content/data"),\n        Path.cwd() / "predict-ai-model-runtime",\n        Path.cwd().parent / "predict-ai-model-runtime",\n        Path("/content/predict-ai-model-runtime"),\n        Path.cwd(),\n    ]\n    for candidate in candidates:\n        if (candidate / "npz_all" / "npz").exists():\n            return candidate\n    raise FileNotFoundError("Could not find npz_all/npz. Put the Kaggle data folder in data/ or update find_data_root().")\n\n\nDATA_ROOT = find_data_root()\nNPZ_ROOT = DATA_ROOT / "npz_all" / "npz"\nCOLLECTIONS = {\n    "layout:xla:default": NPZ_ROOT / "layout" / "xla" / "default",\n    "layout:xla:random": NPZ_ROOT / "layout" / "xla" / "random",\n}\n\nprint("DATA_ROOT:", DATA_ROOT)\nfor name, path in COLLECTIONS.items():\n    print(name, "train", len(list((path / "train").glob("*.npz"))), "valid", len(list((path / "valid").glob("*.npz"))))\n\n\ndef split_files(collection_name, split):\n    return sorted((COLLECTIONS[collection_name] / split).glob("*.npz"))\n\n\ndef get_num_configs(data):\n    if "node_config_feat" in data:\n        return data["node_config_feat"].shape[0]\n    if "config_feat" in data:\n        return data["config_feat"].shape[0]\n    raise KeyError("Could not find config features")\n\n\ndef choose_indices(n_items, max_items=None, seed=RANDOM_SEED):\n    if max_items is None or n_items <= max_items:\n        return np.arange(n_items, dtype=np.int64)\n    local_rng = np.random.default_rng(seed)\n    return np.sort(local_rng.choice(n_items, size=max_items, replace=False)).astype(np.int64)\n\n\ndef choose_runtime_stratified_indices(runtimes, max_items, seed=RANDOM_SEED):\n    runtimes = np.asarray(runtimes, dtype=np.float64)\n    valid_idx = np.flatnonzero(np.isfinite(runtimes) & (runtimes > 0))\n    if max_items is None or len(valid_idx) <= max_items:\n        return valid_idx\n    if len(valid_idx) == 0:\n        raise ValueError("No positive finite training runtimes")\n\n    local_rng = np.random.default_rng(seed)\n    sorted_idx = valid_idx[np.argsort(runtimes[valid_idx])]\n    fastest_count = max(1, int(max_items * 0.20))\n    slowest_count = max(1, int(max_items * 0.10))\n    selected = set(sorted_idx[:fastest_count].tolist())\n    selected.update(sorted_idx[-slowest_count:].tolist())\n\n    remaining = np.array([idx for idx in valid_idx if idx not in selected], dtype=np.int64)\n    budget = max_items - len(selected)\n    if budget > 0 and len(remaining) > 0:\n        chosen = local_rng.choice(remaining, size=min(budget, len(remaining)), replace=False)\n        selected.update(chosen.tolist())\n    return np.array(sorted(selected), dtype=np.int64)\n\n\ndef choose_config_indices(data, split, max_items=None, seed=RANDOM_SEED):\n    n_items = get_num_configs(data)\n    if split == "train" and "config_runtime" in data:\n        return choose_runtime_stratified_indices(data["config_runtime"], max_items, seed=seed)\n    return choose_indices(n_items, max_items=max_items, seed=seed)\n\n\ndef sampled_kendall_score(y_true, y_pred, max_pairs=20000, seed=RANDOM_SEED):\n    n = len(y_true)\n    if n < 2:\n        return np.nan\n    local_rng = np.random.default_rng(seed)\n    i = local_rng.integers(0, n, size=max_pairs)\n    j = local_rng.integers(0, n, size=max_pairs)\n    mask = i != j\n    i, j = i[mask], j[mask]\n    true_order = np.sign(y_true[i] - y_true[j])\n    pred_order = np.sign(y_pred[i] - y_pred[j])\n    useful = true_order != 0\n    if useful.sum() == 0:\n        return np.nan\n    # Prediction ties contribute zero rather than disappearing from the score.\n    return float(np.mean(true_order[useful] * pred_order[useful]))\n\n\ndef centered_log_runtime(runtimes):\n    log_runtime = np.log1p(np.asarray(runtimes, dtype=np.float64))\n    return log_runtime - np.median(log_runtime)\n\n\ndef infer_model_family(file_stem):\n    """Infer a coarse graph/model family from a TpuGraphs file stem."""\n    stem = str(file_stem).lower()\n    known_families = [\'resnet\', \'bert\', \'albert\', \'inception\', \'efficientnet\', \'mlperf\', \'transformer\', \'retinanet\', \'mask_rcnn\', \'mnasnet\', \'alexnet\', \'openai\', \'shapemask\', \'magenta\', \'brax\', \'ncf\', \'xception\', \'electra\', \'talking-heads\', \'trax\', \'unet\', \'experts\']\n    for family in known_families:\n        if stem.startswith(family) or family in stem:\n            return family\n    if len(stem) >= 24 and all((ch in \'0123456789abcdef\' for ch in stem[:24])):\n        return \'hashed_graph_id\'\n    return stem.split(\'_\')[0].split(\'-\')[0].split(\'.\')[0]\n\ndef select_files_for_split(collection_name, split, max_files=None, seed=RANDOM_SEED):\n    """Select graph files, using graph-family stratification for training caps."""\n    files = split_files(collection_name, split)\n    if max_files is None or len(files) <= max_files:\n        return files\n    if split != \'train\':\n        return files[:max_files]\n    grouped = {}\n    for file_path in files:\n        grouped.setdefault(infer_model_family(file_path.stem), []).append(file_path)\n    local_rng = np.random.default_rng(seed)\n    for family_files in grouped.values():\n        local_rng.shuffle(family_files)\n    selected = []\n    families = sorted(grouped, key=lambda family: len(grouped[family]))\n    while len(selected) < max_files and families:\n        progressed = False\n        for family in families:\n            if grouped[family] and len(selected) < max_files:\n                selected.append(grouped[family].pop(0))\n                progressed = True\n        if not progressed:\n            break\n    return sorted(selected)\n\ndef show_training_coverage(collection_name, selected):\n    available = split_files(collection_name, "train")\n    counts = pd.Series([infer_model_family(p.stem) for p in available]).value_counts()\n    chosen = pd.Series([infer_model_family(p.stem) for p in selected]).value_counts()\n    table = pd.DataFrame({"available_graphs": counts, "selected_graphs": chosen}).fillna(0).astype(int)\n    print(collection_name, "training family coverage (inferred from filenames):")\n    display(table.sort_index())\n    return table\n\n\ndef make_validation_manifest(model, files):\n    manifest = []\n    for i, path in enumerate(files):\n        # Only the count is used; sampling never inspects runtime values.\n        with np.load(path, allow_pickle=False) as archive:\n            n_configs = len(archive["config_runtime"])\n        indices = choose_indices(n_configs, GNN_MAX_VALID_CONFIGS_PER_FILE, RANDOM_SEED + i)\n        stat = Path(path).stat()\n        manifest.append({"path": str(Path(path).resolve()), "file": Path(path).stem,\n                         "source_size": stat.st_size, "source_mtime_ns": stat.st_mtime_ns,\n                         "config_indices": indices.tolist()})\n    return manifest\n\n\ndef validation_indices(collection_name, path, n_configs, seed):\n    records = FIXED_VALIDATION.get(collection_name)\n    if records is None:\n        return choose_indices(n_configs, GNN_MAX_VALID_CONFIGS_PER_FILE, seed)\n    match = [r for r in records if r["path"] == str(Path(path).resolve())]\n    if len(match) != 1:\n        raise ValueError(f"File is missing/duplicated in validation manifest: {path}")\n    stat = Path(path).stat()\n    if stat.st_size != match[0]["source_size"] or stat.st_mtime_ns != match[0]["source_mtime_ns"]:\n        raise ValueError(f"Validation source changed since manifest creation: {path}")\n    return np.asarray(match[0]["config_indices"], dtype=np.int64)\n\n\ndef build_adjacency(edge_index, node_count):\n    neighbors = [set() for _ in range(node_count)]\n    for src, dst in np.asarray(edge_index, dtype=np.int64):\n        if 0 <= src < node_count and 0 <= dst < node_count:\n            neighbors[int(src)].add(int(dst))\n            neighbors[int(dst)].add(int(src))\n    return neighbors\n\ndef khop_subgraph_nodes(edge_index, node_count, seed_nodes, hops=2, max_nodes=512):\n    neighbors = build_adjacency(edge_index, node_count)\n    seed_nodes = [int(n) for n in seed_nodes if 0 <= int(n) < node_count]\n    if not seed_nodes:\n        return np.arange(min(node_count, max_nodes), dtype=np.int64)\n    reached = set(seed_nodes)\n    frontier = set(seed_nodes)\n    ordered = list(dict.fromkeys(seed_nodes))\n    for _ in range(hops):\n        next_frontier = set()\n        for node in sorted(frontier):\n            for nbr in sorted(neighbors[node]):\n                if nbr not in reached:\n                    reached.add(nbr)\n                    next_frontier.add(nbr)\n                    ordered.append(nbr)\n        frontier = next_frontier\n        if not frontier:\n            break\n    if len(ordered) > max_nodes:\n        seed_set = list(dict.fromkeys(seed_nodes))\n        remaining = [node for node in ordered if node not in set(seed_set)]\n        ordered = seed_set + remaining[:max(0, max_nodes - len(seed_set))]\n    return np.array(sorted(ordered), dtype=np.int64)\n\ndef induced_edges(edge_index, kept_nodes):\n    kept_nodes = np.asarray(kept_nodes, dtype=np.int64)\n    old_to_new = {int(old): i for i, old in enumerate(kept_nodes)}\n    src_list = []\n    dst_list = []\n    for src, dst in np.asarray(edge_index, dtype=np.int64):\n        if int(src) in old_to_new and int(dst) in old_to_new:\n            src_list.append(old_to_new[int(src)])\n            dst_list.append(old_to_new[int(dst)])\n            src_list.append(old_to_new[int(dst)])\n            dst_list.append(old_to_new[int(src)])\n    for i in range(len(kept_nodes)):\n        src_list.append(i)\n        dst_list.append(i)\n    return (np.asarray(src_list, dtype=np.int64), np.asarray(dst_list, dtype=np.int64), old_to_new)\n\ndef inspect_subgraph_sizes(max_files=5):\n    rows = []\n    for collection_name in GNN_COLLECTIONS:\n        for file_path in split_files(collection_name, \'train\')[:max_files]:\n            with np.load(file_path) as data:\n                node_count = int(data[\'node_feat\'].shape[0])\n                node_config_ids = np.asarray(data[\'node_config_ids\'], dtype=np.int64)\n                kept = khop_subgraph_nodes(data[\'edge_index\'], node_count, node_config_ids, hops=GNN_SUBGRAPH_HOPS, max_nodes=GNN_MAX_SUBGRAPH_NODES)\n                rows.append({\'collection\': collection_name, \'file\': file_path.stem, \'node_count\': node_count, \'configurable_nodes\': len(node_config_ids), \'subgraph_nodes\': len(kept)})\n    return pd.DataFrame(rows)\n\nclass GraphSAGERanker(nn.Module):\n    def __init__(self, node_dim, config_dim, hidden_dim=GNN_HIDDEN_DIM, opcode_vocab_size=256, opcode_emb_dim=16):\n        super().__init__()\n        self.opcode_embedding = nn.Embedding(opcode_vocab_size, opcode_emb_dim)\n        self.input_proj = nn.Linear(node_dim + config_dim + opcode_emb_dim, hidden_dim)\n        self.sage1 = nn.Linear(hidden_dim * 2, hidden_dim)\n        self.sage2 = nn.Linear(hidden_dim * 2, hidden_dim)\n        self.head = nn.Sequential(\n            nn.Linear(hidden_dim * 4, hidden_dim),\n            nn.ReLU(),\n            nn.Dropout(0.05),\n            nn.Linear(hidden_dim, 1),\n        )\n\n    def sage_step(self, h, edge_src, edge_dst, degree, layer):\n        neigh = torch.zeros_like(h)\n        neigh.index_add_(1, edge_dst, h[:, edge_src, :])\n        neigh = neigh / degree.view(1, -1, 1).clamp_min(1.0)\n        return F.relu(layer(torch.cat([h, neigh], dim=-1))) + h\n\n    def forward(self, base_node, opcode, config_node, edge_src, edge_dst, degree, config_mask):\n        batch_size = config_node.shape[0]\n        base = base_node.unsqueeze(0).expand(batch_size, -1, -1)\n        opcode_emb = self.opcode_embedding(opcode).unsqueeze(0).expand(batch_size, -1, -1)\n        h = F.relu(self.input_proj(torch.cat([base, config_node, opcode_emb], dim=-1)))\n        h = self.sage_step(h, edge_src, edge_dst, degree, self.sage1)\n        h = self.sage_step(h, edge_src, edge_dst, degree, self.sage2)\n\n        global_pool = torch.cat([h.mean(dim=1), h.max(dim=1).values], dim=-1)\n        mask = config_mask.view(1, -1, 1).float()\n        denom = mask.sum(dim=1).clamp_min(1.0)\n        config_mean = (h * mask).sum(dim=1) / denom\n        config_max = h.masked_fill(mask == 0, -1e9).max(dim=1).values\n        config_max = torch.where((mask.sum(dim=1) > 0), config_max, torch.zeros_like(config_max))\n        config_pool = torch.cat([config_mean, config_max], dim=-1)\n        return self.head(torch.cat([global_pool, config_pool], dim=-1)).squeeze(-1)\n\n\ndef pairwise_margin_ranking_loss(scores, runtimes, margin=PAIR_MARGIN, min_gap=PAIR_MIN_LOG_RUNTIME_GAP, max_pairs=PAIR_MAX_PAIRS):\n    log_runtime = torch.log1p(runtimes)\n    diff = log_runtime.view(-1, 1) - log_runtime.view(1, -1)\n    pairs = torch.nonzero(diff < -min_gap, as_tuple=False)\n    if pairs.numel() == 0:\n        return F.smooth_l1_loss(scores, log_runtime - log_runtime.median())\n    if len(pairs) > max_pairs:\n        idx = torch.randperm(len(pairs), device=scores.device)[:max_pairs]\n        pairs = pairs[idx]\n    fast = pairs[:, 0]\n    slow = pairs[:, 1]\n    return F.relu(margin + scores[fast] - scores[slow]).mean()\n\n\nclass PreparedGraphCache:\n    """Compact sampled rows plus an LRU of reusable static graph tensors."""\n    def __init__(self, root=GNN_CACHE_DIR, max_mb=GNN_GRAPH_CACHE_MB):\n        self.root = Path(root)\n        self.root.mkdir(parents=True, exist_ok=True)\n        self.max_bytes = int(max_mb * 1024**2)\n        self.entries = OrderedDict()\n        self.selections = {}\n        self.selection_kinds = {}\n        self.bytes = 0\n        self.extractions = 0\n        self.graph_preparations = 0\n        self.prefetch_seconds = 0.0\n\n    def _resolved(self, file_path):\n        return str(Path(file_path).resolve())\n\n    def register_selection(self, file_path, indices, kind):\n        path = Path(file_path).resolve()\n        indices = np.asarray(indices, dtype=np.int64)\n        total = num_configs_in_npz(path)\n        if indices.ndim != 1 or len(np.unique(indices)) != len(indices):\n            raise ValueError(f"Invalid or duplicate cached indices: {path}")\n        if (indices < 0).any() or (indices >= total).any():\n            raise IndexError(f"Cached configuration index outside [0, {total}): {path}")\n        self.selections[self._resolved(path)] = indices.copy()\n        self.selection_kinds[self._resolved(path)] = str(kind)\n\n    def _selection(self, file_path):\n        key = self._resolved(file_path)\n        if key not in self.selections:\n            raise KeyError(f"No compact selection registered for {file_path}")\n        return self.selections[key]\n\n    def _materialize(self, file_path):\n        indices = self._selection(file_path)\n        before = SAMPLED_CONFIG_CACHE.misses\n        values = SAMPLED_CONFIG_CACHE.materialize(file_path, "node_config_feat", indices)\n        self.extractions += SAMPLED_CONFIG_CACHE.misses - before\n        return values\n\n    def prepare(self, paths, ranker):\n        """Prefetch one sampled array while static graph data for the current file is prepared."""\n        paths = list(paths)\n        if not paths:\n            return\n        started = time.perf_counter()\n        with ThreadPoolExecutor(max_workers=GNN_PREFETCH_WORKERS) as executor:\n            future = executor.submit(self._materialize, paths[0])\n            for position, path in enumerate(paths):\n                configs = future.result()\n                if position + 1 < len(paths):\n                    future = executor.submit(self._materialize, paths[position + 1])\n                yield self.get(path, ranker, prefetched_configs=configs)\n        self.prefetch_seconds += time.perf_counter() - started\n\n    def get(self, file_path, ranker, prefetched_configs=None):\n        path = Path(file_path).resolve()\n        indices = self._selection(path)\n        stat = path.stat()\n        selection_digest = hashlib.sha256(indices.tobytes()).hexdigest()[:16]\n        token = f"{path}|{stat.st_size}|{stat.st_mtime_ns}|{selection_digest}"\n        graph_key = (hashlib.sha256(token.encode()).hexdigest()[:24],\n                     GNN_SUBGRAPH_HOPS, GNN_MAX_SUBGRAPH_NODES)\n        if graph_key in self.entries:\n            self.entries.move_to_end(graph_key)\n            return self.entries[graph_key]\n\n        configs = (prefetched_configs if prefetched_configs is not None\n                   else self._materialize(path))\n        data = load_npz_without_node_configs(path)\n        runtime = data.get("config_runtime")\n        selected_runtime = None if runtime is None else np.asarray(runtime[indices])\n        data["node_config_feat"] = configs\n        ranker.ensure_model(data)\n        graph = ranker.prepare_graph(data)\n        graph = tuple(x.cpu() if torch.is_tensor(x) else x for x in graph)\n        nbytes = sum(x.numel() * x.element_size() if torch.is_tensor(x) else x.nbytes for x in graph)\n        # Disk-backed memmaps do not consume cache-sized resident RAM. A low-disk\n        # fallback is a normal ndarray, so include it in the same bounded LRU.\n        config_ram_bytes = 0 if isinstance(configs, np.memmap) else int(configs.nbytes)\n        nbytes += config_ram_bytes\n        if selected_runtime is not None:\n            nbytes += selected_runtime.nbytes\n        entry = {"graph": graph, "configs": configs, "runtime": selected_runtime,\n                 "source_indices": indices.copy(), "total_configs": num_configs_in_npz(path),\n                 "config_ram_bytes": config_ram_bytes, "bytes": nbytes}\n        self.graph_preparations += 1\n        while self.entries and self.bytes + nbytes > self.max_bytes:\n            _, old = self.entries.popitem(last=False)\n            self.bytes -= old["bytes"]\n        if nbytes <= self.max_bytes:\n            self.entries[graph_key] = entry\n            self.bytes += nbytes\n        return entry\n\n    def clear(self, remove_disk=False):\n        selections = [(path, indices.copy()) for path, indices in self.selections.items()]\n        self.entries.clear()\n        self.bytes = 0\n        gc.collect()\n        if remove_disk:\n            for path, indices in selections:\n                SAMPLED_CONFIG_CACHE.remove(path, "node_config_feat", indices)\n        self.selections.clear()\n        self.selection_kinds.clear()\n\n\n\nclass LayoutGNNRanker:\n    def __init__(self):\n        self.device = DEVICE\n        self.model = None\n        self.node_dim = None\n        self.config_dim = None\n        self.history = []\n        self.cache = PreparedGraphCache()\n        self.best_epoch = None\n\n    def base_node_features(self, data, kept_nodes):\n        node_feat = np.asarray(data["node_feat"][kept_nodes], dtype=np.float32)\n        node_feat = np.log1p(np.maximum(node_feat, 0.0))\n        return (node_feat - node_feat.mean(axis=0, keepdims=True)) / (node_feat.std(axis=0, keepdims=True) + 1e-6)\n\n    def prepare_graph(self, data):\n        node_count = int(data["node_feat"].shape[0])\n        raw_config_nodes = np.asarray(data["node_config_ids"], dtype=np.int64)\n        kept_nodes = khop_subgraph_nodes(data["edge_index"], node_count, raw_config_nodes, hops=GNN_SUBGRAPH_HOPS, max_nodes=GNN_MAX_SUBGRAPH_NODES)\n        edge_src_np, edge_dst_np, old_to_new = induced_edges(data["edge_index"], kept_nodes)\n\n        config_local_pairs = [(pos, old_to_new[int(old)]) for pos, old in enumerate(raw_config_nodes) if int(old) in old_to_new]\n        config_positions = np.array([p for p, _ in config_local_pairs], dtype=np.int64)\n        config_local_nodes = np.array([n for _, n in config_local_pairs], dtype=np.int64)\n        config_mask = np.zeros(len(kept_nodes), dtype=np.float32)\n        config_mask[config_local_nodes] = 1.0\n\n        degree = np.bincount(edge_dst_np, minlength=len(kept_nodes)).astype(np.float32)\n        base_node = torch.tensor(self.base_node_features(data, kept_nodes), dtype=torch.float32, device=self.device)\n        opcode = np.clip(np.asarray(data["node_opcode"][kept_nodes], dtype=np.int64), 0, 255)\n        opcode = torch.tensor(opcode, dtype=torch.long, device=self.device)\n        edge_src = torch.tensor(edge_src_np, dtype=torch.long, device=self.device)\n        edge_dst = torch.tensor(edge_dst_np, dtype=torch.long, device=self.device)\n        degree = torch.tensor(degree, dtype=torch.float32, device=self.device)\n        config_mask = torch.tensor(config_mask, dtype=torch.float32, device=self.device)\n        return base_node, opcode, edge_src, edge_dst, degree, config_mask, config_positions, config_local_nodes\n\n    def ensure_model_dimensions(self, node_dim, config_dim):\n        node_dim = int(node_dim)\n        config_dim = int(config_dim)\n        if self.model is None:\n            self.node_dim = node_dim\n            self.config_dim = config_dim\n            self.model = GraphSAGERanker(node_dim=node_dim, config_dim=config_dim).to(self.device)\n        if node_dim != self.node_dim or config_dim != self.config_dim:\n            raise ValueError("Inconsistent node/config feature dimensions")\n        return self.model\n\n    def ensure_model(self, data):\n        return self.ensure_model_dimensions(data["node_feat"].shape[1], data["node_config_feat"].shape[2])\n\n\n    def config_tensor(self, data, config_indices, n_nodes, config_positions, config_local_nodes):\n        selected = np.asarray(data["node_config_feat"][config_indices], dtype=np.float32)\n        selected = np.where(selected == -1, 0.0, selected)\n        config_node = np.zeros((len(config_indices), n_nodes, selected.shape[2]), dtype=np.float32)\n        if len(config_positions):\n            config_node[:, config_local_nodes, :] = selected[:, config_positions, :]\n        return torch.tensor(config_node, dtype=torch.float32, device=self.device)\n\n    def device_graph(self, entry):\n        return tuple(x.to(self.device) if torch.is_tensor(x) else x for x in entry["graph"])\n\n    def cached_config_tensor(self, entry, config_indices, graph):\n        base_node, _, _, _, _, _, positions, local_nodes = graph\n        # Select configurations AND retained nodes before converting to float32.\n        # Only this batch is copied out of the memory map.\n        selected = np.asarray(entry["configs"][np.ix_(config_indices, positions)], dtype=np.float32)\n        selected = np.where(selected == -1, 0.0, selected)\n        padded = np.zeros((len(config_indices), len(base_node), entry["configs"].shape[2]), dtype=np.float32)\n        padded[:, local_nodes, :] = selected\n        return torch.from_numpy(padded).to(self.device)\n\n    def forward_batch(self, entry, indices, graph):\n        base, opcode, src, dst, degree, mask, _, _ = graph\n        config = self.cached_config_tensor(entry, indices, graph)\n        return self.model(base, opcode, config, src, dst, degree, mask)\n\n    def fit(self, collection_name, files, valid_files=None):\n        files = list(files)\n        valid_files = list(valid_files or [])\n        if not files:\n            raise ValueError(f"No training files for {collection_name}")\n        if {Path(p).resolve() for p in files} & {Path(p).resolve() for p in valid_files}:\n            raise ValueError("Training and validation files overlap")\n        torch.manual_seed(RANDOM_SEED)\n\n        # Preserve the original samples exactly, but choose them before expanding configuration data.\n        for file_id, path in enumerate(files):\n            with np.load(path, allow_pickle=False) as archive:\n                runtime = np.asarray(archive["config_runtime"])\n            selected = choose_runtime_stratified_indices(\n                runtime, GNN_MAX_TRAIN_CONFIGS_PER_FILE, seed=RANDOM_SEED + file_id\n            )\n            self.cache.register_selection(path, selected, "train")\n        for file_id, path in enumerate(valid_files):\n            selected = validation_indices(\n                collection_name, path, num_configs_in_npz(path), RANDOM_SEED + file_id\n            )\n            self.cache.register_selection(path, selected, "valid")\n\n        started = time.perf_counter()\n        for _ in tqdm(self.cache.prepare(files + valid_files, self),\n                      total=len(files) + len(valid_files), desc="Prepare compact graph cache", unit="file"):\n            pass\n        preparation_seconds = time.perf_counter() - started\n        print(f"Compact cache preparation: {preparation_seconds:.2f}s; "\n              f"sample writes={self.cache.extractions}; static RAM={self.cache.bytes / 1024**2:.1f} MiB; "\n              f"sample disk={SAMPLED_CONFIG_CACHE.disk_bytes() / 1024**2:.1f} MiB")\n        optimizer = torch.optim.AdamW(self.model.parameters(), lr=GNN_LR, weight_decay=1e-4)\n        best_score, best_state, stale = -np.inf, None, 0\n        run_dir = GNN_OUTPUT_DIR / collection_name.replace(":", "_")\n        run_dir.mkdir(parents=True, exist_ok=True)\n\n        for epoch in range(1, GNN_EPOCHS + 1):\n            self.model.train()\n            losses, examples = [], 0\n            prepare_seconds = 0.0\n            tensor_seconds = 0.0\n            step_seconds = 0.0\n            epoch_start = time.perf_counter()\n            progress = tqdm(files, desc=f"{collection_name} epoch {epoch}/{GNN_EPOCHS}", unit="file")\n            for file_id, file_path in enumerate(progress):\n                t = time.perf_counter()\n                entry = self.cache.get(file_path, self)\n                graph = self.device_graph(entry)\n                if entry["runtime"] is None:\n                    raise ValueError(f"Training runtime labels missing: {file_path}")\n                local_indices = np.arange(len(entry["configs"]), dtype=np.int64)\n                np.random.default_rng(RANDOM_SEED + epoch + file_id).shuffle(local_indices)\n                prepare_seconds += time.perf_counter() - t\n                for start in range(0, len(local_indices), GNN_BATCH_SIZE):\n                    batch = local_indices[start:start + GNN_BATCH_SIZE]\n                    t = time.perf_counter()\n                    config = self.cached_config_tensor(entry, batch, graph)\n                    runtime = torch.as_tensor(np.asarray(entry["runtime"][batch], dtype=np.float32), device=self.device)\n                    tensor_seconds += time.perf_counter() - t\n                    t = time.perf_counter()\n                    base, opcode, src, dst, degree, mask, _, _ = graph\n                    scores = self.model(base, opcode, config, src, dst, degree, mask)\n                    loss = pairwise_margin_ranking_loss(scores, runtime)\n                    if not torch.isfinite(loss):\n                        raise ValueError(f"Nonfinite training loss: {file_path}")\n                    optimizer.zero_grad(set_to_none=True)\n                    loss.backward()\n                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)\n                    optimizer.step()\n                    losses.append(float(loss.detach().cpu()))\n                    step_seconds += time.perf_counter() - t\n                    examples += len(batch)\n                progress.set_postfix(loss=np.mean(losses[-10:]))\n            train_seconds = time.perf_counter() - epoch_start\n            row = {"epoch": epoch, "loss": float(np.mean(losses)),\n                   "cache_preparation_seconds": preparation_seconds if epoch == 1 else 0.0,\n                   "graph_lookup_seconds": prepare_seconds, "batch_tensor_seconds": tensor_seconds,\n                   "model_step_seconds": step_seconds, "train_seconds": train_seconds,\n                   "configs_per_second": examples / max(train_seconds, 1e-9),\n                   "sample_cache_mib": SAMPLED_CONFIG_CACHE.disk_bytes() / 1024**2}\n            if valid_files and (epoch % GNN_VALIDATE_EVERY == 0 or epoch == GNN_EPOCHS):\n                t = time.perf_counter()\n                valid = validate_model(self, collection_name, files=valid_files)\n                score = float(valid["ranking_score"].mean())\n                row.update(validation_score=score, validation_seconds=time.perf_counter() - t)\n                if np.isfinite(score) and score > best_score + GNN_MIN_DELTA:\n                    best_score, stale = score, 0\n                    self.best_epoch = epoch\n                    best_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}\n                    torch.save({"model_state": best_state, "epoch": epoch, "validation_score": score,\n                                "node_dim": self.node_dim, "config_dim": self.config_dim,\n                                "hidden_dim": GNN_HIDDEN_DIM, "seed": RANDOM_SEED,\n                                "subgraph_hops": GNN_SUBGRAPH_HOPS, "max_nodes": GNN_MAX_SUBGRAPH_NODES,\n                                "batch_size": GNN_BATCH_SIZE,\n                                "train_files": [str(p) for p in files],\n                                "valid_files": [str(p) for p in valid_files]}, run_dir / "best.pt")\n                else:\n                    stale += 1\n            self.history.append(row)\n            pd.DataFrame(self.history).to_csv(run_dir / "history.csv", index=False)\n            print(row)\n            if valid_files and stale >= GNN_PATIENCE:\n                print(f"Early stopping; best epoch: {self.best_epoch}")\n                break\n        if best_state is not None:\n            self.model.load_state_dict(best_state)\n        self.model.eval()\n        return self\n\n\n    def predict_file(self, file_path, split="valid", max_configs=None, seed=RANDOM_SEED, config_indices=None):\n        if self.model is None:\n            raise RuntimeError("Fit or load a model before prediction")\n        self.model.eval()\n        resolved = self.cache._resolved(file_path)\n        if resolved in self.cache.selections:\n            entry = self.cache.get(file_path, self)\n            source = entry["source_indices"]\n            requested = (source if config_indices is None else np.asarray(config_indices, dtype=np.int64))\n            if config_indices is None and max_configs is not None:\n                requested = source[choose_indices(len(source), max_items=max_configs, seed=seed)]\n            lookup = {int(value): position for position, value in enumerate(source)}\n            if requested.ndim != 1 or len(np.unique(requested)) != len(requested) or any(int(i) not in lookup for i in requested):\n                raise ValueError("Prediction indices are not present in the registered compact sample")\n            local = np.asarray([lookup[int(value)] for value in requested], dtype=np.int64)\n            graph = self.device_graph(entry)\n            preds = []\n            with torch.inference_mode():\n                for start in range(0, len(local), GNN_PREDICT_BATCH_SIZE):\n                    batch = local[start:start + GNN_PREDICT_BATCH_SIZE]\n                    preds.append(self.forward_batch(entry, batch, graph).cpu().numpy())\n            return requested, np.concatenate(preds) if preds else np.array([], dtype=np.float64)\n\n        # Full test ranking: keep one compressed member open and never create a full expanded file.\n        data = load_npz_without_node_configs(file_path)\n        shape, _, _ = npy_member_metadata(file_path, "node_config_feat")\n        self.ensure_model_dimensions(data["node_feat"].shape[1], shape[2])\n        graph = tuple(x.cpu() if torch.is_tensor(x) else x for x in self.prepare_graph(data))\n        graph = tuple(x.to(self.device) if torch.is_tensor(x) else x for x in graph)\n        if config_indices is not None or max_configs is not None:\n            requested = (np.asarray(config_indices, dtype=np.int64) if config_indices is not None\n                         else choose_indices(shape[0], max_items=max_configs, seed=seed))\n            rows = read_npz_rows(file_path, "node_config_feat", requested)\n            batches = ((requested[start:start + GNN_PREDICT_BATCH_SIZE],\n                        rows[start:start + GNN_PREDICT_BATCH_SIZE])\n                       for start in range(0, len(requested), GNN_PREDICT_BATCH_SIZE))\n        else:\n            batches = iter_npz_row_batches(file_path, "node_config_feat", GNN_PREDICT_BATCH_SIZE)\n        output_indices, preds = [], []\n        with torch.inference_mode():\n            for source_indices, rows in batches:\n                transient = {"configs": rows}\n                local = np.arange(len(rows), dtype=np.int64)\n                preds.append(self.forward_batch(transient, local, graph).cpu().numpy())\n                output_indices.append(np.asarray(source_indices, dtype=np.int64))\n        indices = np.concatenate(output_indices) if output_indices else np.array([], dtype=np.int64)\n        values = np.concatenate(preds) if preds else np.array([], dtype=np.float64)\n        print(f"Streamed {len(indices)} configurations from {Path(file_path).name}; persistent full-array disk use: 0 MiB")\n        return indices, values\n\n\n\ndef validate_model(model, collection_name, files=None):\n    if files is None:\n        files = split_files(collection_name, \'valid\')[:GNN_MAX_VALID_FILES]\n    rows = []\n    for i, file_path in enumerate(files):\n        entry = model.cache.get(file_path, model)\n        wanted = entry[\'source_indices\']\n        indices, pred = model.predict_file(file_path, config_indices=wanted)\n        if entry[\'runtime\'] is None:\n            raise ValueError(f\'Validation runtime labels missing: {file_path}\')\n        truth = np.asarray(entry[\'runtime\'], dtype=np.float64)\n        if not (np.isfinite(truth).all() and (truth > 0).all() and np.isfinite(pred).all()):\n            raise ValueError(f\'Invalid validation runtimes or predictions: {file_path}\')\n        rows.append({\'collection\': collection_name, \'file\': file_path.stem,\n                     \'n_configs\': len(truth),\n                     \'ranking_score\': sampled_kendall_score(truth, pred, seed=RANDOM_SEED + i)})\n    return pd.DataFrame(rows)\n\n\ndef make_baseline_feature_namespace():\n    namespace = {\'np\': np, \'pd\': pd}\n    exec(BASELINE_FEATURE_SOURCE, namespace)\n    return namespace\nBASELINE_FEATURE_SOURCE = \'import time\\nimport zlib\\nFEATURE_HASH_BINS = 128\\nWL_DEPTH = 3\\nFEATURE_EXPERIMENTS = {\\\'simple_summary_ablation\\\': {\\\'use_degree_features\\\': False, \\\'use_dag_depth_features\\\': False, \\\'use_opcode_transition_features\\\': False, \\\'use_repeated_subgraph_features\\\': False, \\\'use_wl_features\\\': False, \\\'use_layout_local_graph_features\\\': False}, \\\'paper_mlp_baseline\\\': {\\\'use_degree_features\\\': False, \\\'use_dag_depth_features\\\': False, \\\'use_opcode_transition_features\\\': False, \\\'use_repeated_subgraph_features\\\': False, \\\'use_wl_features\\\': False, \\\'use_layout_local_graph_features\\\': False}, \\\'repeated_subgraph\\\': {\\\'use_degree_features\\\': True, \\\'use_dag_depth_features\\\': True, \\\'use_opcode_transition_features\\\': True, \\\'use_repeated_subgraph_features\\\': True, \\\'use_wl_features\\\': False, \\\'use_layout_local_graph_features\\\': True}, \\\'wl_fingerprint\\\': {\\\'use_degree_features\\\': True, \\\'use_dag_depth_features\\\': True, \\\'use_opcode_transition_features\\\': False, \\\'use_repeated_subgraph_features\\\': False, \\\'use_wl_features\\\': True, \\\'use_layout_local_graph_features\\\': True}, \\\'combined_compact_graph\\\': {\\\'use_degree_features\\\': True, \\\'use_dag_depth_features\\\': True, \\\'use_opcode_transition_features\\\': True, \\\'use_repeated_subgraph_features\\\': True, \\\'use_wl_features\\\': True, \\\'use_layout_local_graph_features\\\': True}}\\nDEFAULT_FEATURE_PROFILE_NAME = \\\'combined_compact_graph\\\'\\nACTIVE_FEATURE_SETTINGS = FEATURE_EXPERIMENTS[DEFAULT_FEATURE_PROFILE_NAME]\\n\\ndef merge_feature_settings(feature_settings=None):\\n    settings = FEATURE_EXPERIMENTS[\\\'simple_summary_ablation\\\'].copy()\\n    if feature_settings is None:\\n        settings.update(ACTIVE_FEATURE_SETTINGS)\\n    else:\\n        settings.update(feature_settings)\\n    return settings\\n\\ndef stable_hash_to_bin(value, n_bins=FEATURE_HASH_BINS):\\n    """Deterministic hash binning for graph patterns."""\\n    if not isinstance(value, bytes):\\n        value = str(value).encode(\\\'utf-8\\\')\\n    return zlib.crc32(value) % n_bins\\n\\ndef safe_numeric_stats(prefix, arr):\\n    """Small aggregate stats. These are cheap and work for arrays of different shapes."""\\n    arr = np.asarray(arr)\\n    values = arr[np.isfinite(arr)] if np.issubdtype(arr.dtype, np.number) else np.array([])\\n    if values.size == 0:\\n        return {f\\\'{prefix}_mean\\\': 0.0, f\\\'{prefix}_std\\\': 0.0, f\\\'{prefix}_min\\\': 0.0, f\\\'{prefix}_max\\\': 0.0}\\n    return {f\\\'{prefix}_mean\\\': float(values.mean()), f\\\'{prefix}_std\\\': float(values.std()), f\\\'{prefix}_min\\\': float(values.min()), f\\\'{prefix}_max\\\': float(values.max())}\\n\\ndef distribution_stats(prefix, values):\\n    """Fixed summary columns for one-dimensional graph statistics."""\\n    values = np.asarray(values, dtype=np.float64)\\n    if values.size == 0:\\n        values = np.array([0.0])\\n    stats = safe_numeric_stats(prefix, values)\\n    for percentile in [10, 25, 50, 75, 90]:\\n        stats[f\\\'{prefix}_p{percentile}\\\'] = float(np.percentile(values, percentile))\\n    return stats\\n\\ndef build_adjacency(edge_index, node_count):\\n    """Return incoming and outgoing adjacency lists for a directed graph."""\\n    incoming = [[] for _ in range(node_count)]\\n    outgoing = [[] for _ in range(node_count)]\\n    for src, dst in np.asarray(edge_index, dtype=np.int64):\\n        if 0 <= src < node_count and 0 <= dst < node_count:\\n            outgoing[int(src)].append(int(dst))\\n            incoming[int(dst)].append(int(src))\\n    return (incoming, outgoing)\\n\\ndef degree_features(edge_index, node_count):\\n    """Degree summaries preserve more graph structure than edge count alone."""\\n    incoming, outgoing = build_adjacency(edge_index, node_count)\\n    in_degree = np.array([len(nodes) for nodes in incoming], dtype=np.float64)\\n    out_degree = np.array([len(nodes) for nodes in outgoing], dtype=np.float64)\\n    total_degree = in_degree + out_degree\\n    features = {}\\n    features.update(distribution_stats(\\\'in_degree\\\', in_degree))\\n    features.update(distribution_stats(\\\'out_degree\\\', out_degree))\\n    features.update(distribution_stats(\\\'total_degree\\\', total_degree))\\n    features[\\\'source_node_frac\\\'] = float(np.mean(in_degree == 0)) if node_count else 0.0\\n    features[\\\'sink_node_frac\\\'] = float(np.mean(out_degree == 0)) if node_count else 0.0\\n    return features\\n\\ndef longest_dag_depths(edge_index, node_count, reverse=False):\\n    """Longest-path depth from sources. If cycles appear, unresolved nodes stay at zero."""\\n    if node_count == 0:\\n        return np.array([], dtype=np.float64)\\n    edges = np.asarray(edge_index, dtype=np.int64)\\n    if reverse:\\n        edges = edges[:, [1, 0]]\\n    incoming, outgoing = build_adjacency(edges, node_count)\\n    indegree = np.array([len(nodes) for nodes in incoming], dtype=np.int64)\\n    queue = [i for i, degree in enumerate(indegree) if degree == 0]\\n    depth = np.zeros(node_count, dtype=np.float64)\\n    head = 0\\n    while head < len(queue):\\n        node = queue[head]\\n        head += 1\\n        for nxt in outgoing[node]:\\n            if depth[nxt] < depth[node] + 1:\\n                depth[nxt] = depth[node] + 1\\n            indegree[nxt] -= 1\\n            if indegree[nxt] == 0:\\n                queue.append(nxt)\\n    return depth\\n\\ndef dag_depth_features(edge_index, node_count):\\n    """Summarize where nodes sit in the computation DAG."""\\n    source_depth = longest_dag_depths(edge_index, node_count, reverse=False)\\n    sink_depth = longest_dag_depths(edge_index, node_count, reverse=True)\\n    features = {}\\n    features.update(distribution_stats(\\\'source_depth\\\', source_depth))\\n    features.update(distribution_stats(\\\'sink_depth\\\', sink_depth))\\n    features[\\\'dag_longest_path_estimate\\\'] = float(max(source_depth.max(initial=0.0), sink_depth.max(initial=0.0)))\\n    return features\\n\\ndef normalized_hash_counts(prefix, bin_ids, n_bins=FEATURE_HASH_BINS):\\n    counts = np.bincount(np.asarray(bin_ids, dtype=np.int64), minlength=n_bins)[:n_bins].astype(np.float64)\\n    total = counts.sum()\\n    if total > 0:\\n        counts /= total\\n    return {f\\\'{prefix}_bin_{i}\\\': float(value) for i, value in enumerate(counts)}\\n\\ndef opcode_transition_features(node_opcode, edge_index, n_bins=FEATURE_HASH_BINS):\\n    """Count directed opcode-to-opcode transitions along graph edges."""\\n    node_opcode = np.asarray(node_opcode, dtype=np.int64)\\n    bin_ids = []\\n    for src, dst in np.asarray(edge_index, dtype=np.int64):\\n        if 0 <= src < len(node_opcode) and 0 <= dst < len(node_opcode):\\n            pattern = f\\\'{int(node_opcode[src])}>{int(node_opcode[dst])}\\\'\\n            bin_ids.append(stable_hash_to_bin(pattern, n_bins))\\n    return normalized_hash_counts(\\\'opcode_transition\\\', bin_ids, n_bins=n_bins)\\n\\ndef repeated_subgraph_features(node_opcode, edge_index, n_bins=FEATURE_HASH_BINS, max_neighbors_per_side=16):\\n    """Approximate repeated local subgraphs by hashing opcode neighborhoods."""\\n    node_opcode = np.asarray(node_opcode, dtype=np.int64)\\n    node_count = len(node_opcode)\\n    incoming, outgoing = build_adjacency(edge_index, node_count)\\n    bin_ids = []\\n    raw_patterns = []\\n    for node in range(node_count):\\n        in_ops = sorted((int(node_opcode[n]) for n in incoming[node]))[:max_neighbors_per_side]\\n        out_ops = sorted((int(node_opcode[n]) for n in outgoing[node]))[:max_neighbors_per_side]\\n        pattern = f\\\'op={int(node_opcode[node])}|in={in_ops}|out={out_ops}\\\'\\n        raw_patterns.append(pattern)\\n        bin_ids.append(stable_hash_to_bin(pattern, n_bins))\\n    features = normalized_hash_counts(\\\'repeat_subgraph\\\', bin_ids, n_bins=n_bins)\\n    pattern_counts = pd.Series(raw_patterns).value_counts() if raw_patterns else pd.Series(dtype=np.int64)\\n    features[\\\'repeat_subgraph_unique_frac\\\'] = float(len(pattern_counts) / max(node_count, 1))\\n    features[\\\'repeat_subgraph_max_frac\\\'] = float(pattern_counts.iloc[0] / max(node_count, 1)) if len(pattern_counts) else 0.0\\n    features[\\\'repeat_subgraph_repeated_frac\\\'] = float(np.mean(pattern_counts.to_numpy() > 1)) if len(pattern_counts) else 0.0\\n    return features\\n\\ndef wl_subtree_features(node_opcode, edge_index, depth=WL_DEPTH, n_bins=FEATURE_HASH_BINS):\\n    """Weisfeiler-Lehman subtree count features over opcode-labeled graph nodes."""\\n    node_opcode = np.asarray(node_opcode, dtype=np.int64)\\n    node_count = len(node_opcode)\\n    incoming, outgoing = build_adjacency(edge_index, node_count)\\n    neighbors = [sorted(set(incoming[i] + outgoing[i])) for i in range(node_count)]\\n    labels = [f\\\'op_{int(op)}\\\' for op in node_opcode]\\n    features = {}\\n    for round_id in range(depth + 1):\\n        bin_ids = [stable_hash_to_bin(label, n_bins) for label in labels]\\n        features.update(normalized_hash_counts(f\\\'wl_round_{round_id}\\\', bin_ids, n_bins=n_bins))\\n        if round_id == depth:\\n            break\\n        next_labels = []\\n        for node in range(node_count):\\n            neighbor_labels = sorted((labels[nbr] for nbr in neighbors[node]))\\n            combined = labels[node] + \\\'|\\\' + \\\'|\\\'.join(neighbor_labels)\\n            next_labels.append(str(zlib.crc32(combined.encode(\\\'utf-8\\\'))))\\n        labels = next_labels\\n    return features\\n\\ndef graph_level_features(data, feature_settings=None):\\n    """Features shared by every configuration inside the same graph file."""\\n    settings = merge_feature_settings(feature_settings)\\n    node_feat = data[\\\'node_feat\\\']\\n    node_opcode = data[\\\'node_opcode\\\']\\n    edge_index = data[\\\'edge_index\\\']\\n    node_count = int(node_feat.shape[0])\\n    edge_count = int(edge_index.shape[0])\\n    features = {\\\'node_count\\\': node_count, \\\'edge_count\\\': edge_count, \\\'edge_per_node\\\': edge_count / max(node_count, 1), \\\'opcode_unique\\\': int(np.unique(node_opcode).size), \\\'opcode_mean\\\': float(np.mean(node_opcode)), \\\'opcode_std\\\': float(np.std(node_opcode))}\\n    features.update(safe_numeric_stats(\\\'node_feat\\\', node_feat))\\n    opcode_hist = np.bincount(node_opcode.astype(np.int64), minlength=128)[:128]\\n    opcode_hist = opcode_hist / max(opcode_hist.sum(), 1)\\n    for i, value in enumerate(opcode_hist):\\n        features[f\\\'opcode_hist_{i}\\\'] = float(value)\\n    if settings[\\\'use_degree_features\\\']:\\n        features.update(degree_features(edge_index, node_count))\\n    if settings[\\\'use_dag_depth_features\\\']:\\n        features.update(dag_depth_features(edge_index, node_count))\\n    if settings[\\\'use_opcode_transition_features\\\']:\\n        features.update(opcode_transition_features(node_opcode, edge_index))\\n    if settings[\\\'use_repeated_subgraph_features\\\']:\\n        features.update(repeated_subgraph_features(node_opcode, edge_index))\\n    if settings[\\\'use_wl_features\\\']:\\n        features.update(wl_subtree_features(node_opcode, edge_index))\\n    return features\\n\\ndef masked_mean_and_std(mask, values):\\n    """Vectorized mean/std of node-level values over valid nodes for each config."""\\n    mask = np.asarray(mask, dtype=np.float64)\\n    values = np.asarray(values, dtype=np.float64)\\n    counts = np.maximum(mask.sum(axis=1), 1.0)\\n    mean = mask @ values / counts\\n    second = mask @ values ** 2 / counts\\n    std = np.sqrt(np.maximum(second - mean ** 2, 0.0))\\n    return (mean, std)\\n\\ndef layout_config_local_graph_features(data, config_indices):\\n    """Graph-position summaries around layout-configurable nodes."""\\n    if \\\'node_config_feat\\\' not in data or \\\'node_config_ids\\\' not in data:\\n        return pd.DataFrame(index=np.arange(len(config_indices)))\\n    node_opcode = np.asarray(data[\\\'node_opcode\\\'], dtype=np.float64)\\n    edge_index = data[\\\'edge_index\\\']\\n    node_count = len(node_opcode)\\n    node_ids = np.asarray(data[\\\'node_config_ids\\\'], dtype=np.int64)\\n    node_ids = np.clip(node_ids, 0, max(node_count - 1, 0))\\n    incoming, outgoing = build_adjacency(edge_index, node_count)\\n    in_degree = np.array([len(nodes) for nodes in incoming], dtype=np.float64)\\n    out_degree = np.array([len(nodes) for nodes in outgoing], dtype=np.float64)\\n    total_degree = in_degree + out_degree\\n    source_depth = longest_dag_depths(edge_index, node_count, reverse=False)\\n    sink_depth = longest_dag_depths(edge_index, node_count, reverse=True)\\n    selected = data[\\\'node_config_feat\\\'][config_indices]\\n    valid_node_mask = np.any(selected != -1, axis=2)\\n    selected_no_pad = np.where(selected == -1, 0, selected)\\n    config_value_by_node = selected_no_pad.mean(axis=2)\\n    counts = np.maximum(valid_node_mask.sum(axis=1), 1)\\n    local_values = {\\\'opcode\\\': node_opcode[node_ids], \\\'in_degree\\\': in_degree[node_ids], \\\'out_degree\\\': out_degree[node_ids], \\\'total_degree\\\': total_degree[node_ids], \\\'source_depth\\\': source_depth[node_ids], \\\'sink_depth\\\': sink_depth[node_ids]}\\n    rows = {\\\'layout_local_valid_node_frac\\\': valid_node_mask.mean(axis=1), \\\'layout_local_config_value_mean\\\': config_value_by_node.sum(axis=1) / counts, \\\'layout_local_config_value_std\\\': np.sqrt(np.maximum((config_value_by_node ** 2).sum(axis=1) / counts - (config_value_by_node.sum(axis=1) / counts) ** 2, 0.0))}\\n    mask_float = valid_node_mask.astype(np.float64)\\n    for name, values in local_values.items():\\n        mean, std = masked_mean_and_std(mask_float, values)\\n        rows[f\\\'layout_local_{name}_mean\\\'] = mean\\n        rows[f\\\'layout_local_{name}_std\\\'] = std\\n        rows[f\\\'layout_local_config_x_{name}_mean\\\'] = (config_value_by_node * values.reshape(1, -1)).sum(axis=1) / counts\\n    return pd.DataFrame(rows)\\n\\ndef config_features_from_file(data, collection_name, config_indices=None, feature_settings=None):\\n    """Return one DataFrame row per selected configuration."""\\n    settings = merge_feature_settings(feature_settings)\\n    if \\\'config_feat\\\' in data:\\n        config_feat = data[\\\'config_feat\\\']\\n        if config_indices is None:\\n            config_indices = np.arange(config_feat.shape[0])\\n        selected = config_feat[config_indices]\\n        rows = pd.DataFrame(selected, columns=[f\\\'tile_config_feat_{i}\\\' for i in range(selected.shape[1])])\\n        rows[\\\'config_feat_mean\\\'] = selected.mean(axis=1)\\n        rows[\\\'config_feat_std\\\'] = selected.std(axis=1)\\n        rows[\\\'config_feat_max\\\'] = selected.max(axis=1)\\n        rows[\\\'config_feat_nonzero_frac\\\'] = (selected != 0).mean(axis=1)\\n    else:\\n        node_config_feat = data[\\\'node_config_feat\\\']\\n        if config_indices is None:\\n            config_indices = np.arange(node_config_feat.shape[0])\\n        selected = node_config_feat[config_indices]\\n        padding_frac = (selected == -1).mean(axis=(1, 2))\\n        selected_no_pad = np.where(selected == -1, 0, selected)\\n        pieces = []\\n        for stat_name, values in [(\\\'mean\\\', selected_no_pad.mean(axis=1)), (\\\'std\\\', selected_no_pad.std(axis=1)), (\\\'min\\\', selected_no_pad.min(axis=1)), (\\\'max\\\', selected_no_pad.max(axis=1))]:\\n            pieces.append(pd.DataFrame(values, columns=[f\\\'layout_config_{stat_name}_{i}\\\' for i in range(values.shape[1])]))\\n        rows = pd.concat(pieces, axis=1)\\n        rows[\\\'layout_config_padding_frac\\\'] = padding_frac\\n        rows[\\\'num_configurable_nodes\\\'] = node_config_feat.shape[1]\\n        if settings[\\\'use_layout_local_graph_features\\\']:\\n            rows = pd.concat([rows, layout_config_local_graph_features(data, config_indices)], axis=1)\\n    rows[\\\'config_index\\\'] = config_indices.astype(int)\\n    rows[\\\'is_tile_collection\\\'] = int(collection_name.startswith(\\\'tile\\\'))\\n    rows[\\\'is_layout_collection\\\'] = int(collection_name.startswith(\\\'layout\\\'))\\n    return rows\'\nBASELINE_FEATURES = make_baseline_feature_namespace()\n\ndef baseline_frame(data, collection, indices, settings):\n    """Keep main\'s exact feature names/values, with bounded configuration batches."""\n    graph_features = BASELINE_FEATURES[\'graph_level_features\'](data, settings)\n    pieces = []\n    for start in range(0, len(indices), BASELINE_FEATURE_BATCH_SIZE):\n        batch = indices[start:start + BASELINE_FEATURE_BATCH_SIZE]\n        frame = BASELINE_FEATURES[\'config_features_from_file\'](data, collection, batch, settings)\n        pieces.append(frame)\n    if not pieces:\n        raise ValueError(\'Empty feature selection\')\n    frame = pd.concat(pieces, ignore_index=True)\n    frame = pd.concat([frame, pd.DataFrame([graph_features] * len(frame))], axis=1)\n    numeric = frame.select_dtypes(include=[np.number]).columns\n    frame[numeric] = frame[numeric].replace([np.inf, -np.inf], 0).fillna(0)\n    np.testing.assert_array_equal(frame[\'config_index\'].to_numpy(), indices)\n    return frame\n\ndef baseline_data(path, ranker):\n    entry = ranker.cache.get(path, ranker)\n    data = load_npz_without_node_configs(path)\n    data[\'node_config_feat\'] = entry[\'configs\']\n    return data, entry[\'runtime\'], entry[\'source_indices\']\n\n\ndef saved_baseline_members():\n    import joblib\n    if BASELINE_ARTIFACT_PATH is not None:\n        path = Path(BASELINE_ARTIFACT_PATH)\n        if not path.is_file():\n            raise FileNotFoundError(path)\n    else:\n        candidates = [Path(\'models/ensemble_members_by_collection.joblib\'), Path(\'Eugene/models/ensemble_members_by_collection.joblib\')]\n        existing = list(dict.fromkeys((p.resolve() for p in candidates if p.is_file())))\n        if len(existing) > 1:\n            raise ValueError(\'Multiple saved baselines found; set BASELINE_ARTIFACT_PATH explicitly\')\n        if not existing:\n            return None\n        path = existing[0]\n    artifact = joblib.load(path)\n    if not isinstance(artifact, dict):\n        raise ValueError("Expected main\'s ensemble_members_by_collection dictionary")\n    for collection in GNN_COLLECTIONS:\n        members = artifact.get(collection)\n        if not members or any((not {\'model\', \'feature_columns\', \'feature_settings\'} <= set(m) for m in members)):\n            raise ValueError(f\'Incomplete saved baseline members for {collection}\')\n    print(\'Using saved baseline members:\', path)\n    return artifact\n\ndef train_boosting_references(ranker, collection, train_files):\n    from sklearn.ensemble import HistGradientBoostingRegressor\n    from threadpoolctl import threadpool_limits\n    references = {}\n    for profile in REFERENCE_FEATURE_PROFILES:\n        settings = BASELINE_FEATURES[\'FEATURE_EXPERIMENTS\'][profile]\n        frames, labels, families = ([], [], [])\n        for i, path in enumerate(tqdm(train_files, desc=f\'Reference features: {profile}\')):\n            data, runtime, source_indices = baseline_data(path, ranker)\n            local = np.arange(len(source_indices), dtype=np.int64)\n            frame = baseline_frame(data, collection, local, settings)\n            frame[\'config_index\'] = source_indices\n            frames.append(frame)\n            log_runtime = np.log1p(runtime.astype(np.float64))\n            labels.extend(log_runtime - np.median(log_runtime))\n            families.extend([infer_model_family(path.stem)] * len(local))\n        X = pd.concat(frames, ignore_index=True).fillna(0)\n        family_series = pd.Series(families)\n        weights = 1.0 / family_series.map(family_series.value_counts()).to_numpy()\n        weights /= weights.mean()\n        model = HistGradientBoostingRegressor(loss=\'squared_error\', learning_rate=0.06,\n                    max_iter=250, max_leaf_nodes=31, l2_regularization=0.01,\n                    random_state=RANDOM_SEED)\n        with threadpool_limits(limits=GNN_CPU_THREADS):\n            model.fit(X, np.asarray(labels), sample_weight=weights)\n        name = f\'reference_hgb_{profile}\'\n        references[name] = [{\'model\': model, \'feature_columns\': list(X.columns),\n                             \'feature_settings\': settings, \'experiment\': name}]\n    return references\n\n\ndef predict_baseline_members(data, collection, indices, members, source_indices=None):\n    from threadpoolctl import threadpool_limits\n    predictions = []\n    frames = {}\n    for member in members:\n        key = json.dumps(member[\'feature_settings\'], sort_keys=True)\n        if key not in frames:\n            frame = baseline_frame(data, collection, indices, member[\'feature_settings\'])\n            if source_indices is not None:\n                frame[\'config_index\'] = np.asarray(source_indices, dtype=np.int64)\n            frames[key] = frame\n        frame = frames[key]\n        missing = set(member[\'feature_columns\']) - set(frame.columns)\n        if missing:\n            raise ValueError(f\'Saved model needs unsupported features: {sorted(missing)}\')\n        with threadpool_limits(limits=GNN_CPU_THREADS):\n            pred = np.asarray(member[\'model\'].predict(frame[member[\'feature_columns\']]))\n        if pred.shape != (len(indices),) or not np.isfinite(pred).all():\n            raise ValueError(\'Baseline predictions are invalid or misaligned\')\n        predictions.append(pred)\n    if len(predictions) == 1:\n        return predictions[0]\n    return np.mean([pd.Series(p).rank(method=\'average\').to_numpy() for p in predictions], axis=0)\n\ndef compare_models(ranker, collection, train_files, valid_files, saved=None):\n    references = ({\'saved_main_ensemble\': saved[collection]} if saved is not None\n                  else train_boosting_references(ranker, collection, train_files))\n    rows, predictions = ([], [])\n    for i, path in enumerate(valid_files):\n        data, runtimes, source_indices = baseline_data(path, ranker)\n        local = np.arange(len(source_indices), dtype=np.int64)\n        got, gnn = ranker.predict_file(path, config_indices=source_indices)\n        np.testing.assert_array_equal(got, source_indices)\n        truth = runtimes.astype(np.float64)\n        candidates = {\'gnn\': gnn}\n        candidates.update({name: predict_baseline_members(\n                              data, collection, local, members, source_indices=source_indices)\n                           for name, members in references.items()})\n        predictions.append(pd.DataFrame({\'collection\': collection, \'file\': path.stem,\n                         \'config_index\': source_indices, \'runtime\': truth, **candidates}))\n        for name, pred in candidates.items():\n            rows.append({\'collection\': collection, \'file\': path.stem,\n                         \'family\': infer_model_family(path.stem), \'model\': name,\n                         \'n_configs\': len(source_indices),\n                         \'ranking_score\': sampled_kendall_score(truth, pred, seed=RANDOM_SEED + i)})\n    detail = pd.DataFrame(rows)\n    run_dir = GNN_OUTPUT_DIR / collection.replace(\':\', \'_\')\n    run_dir.mkdir(parents=True, exist_ok=True)\n    detail.to_csv(run_dir / \'comparison_by_graph.csv\', index=False)\n    pd.concat(predictions, ignore_index=True).to_csv(run_dir / \'comparison_predictions.csv\', index=False)\n    table = detail.groupby(\'model\', as_index=False).agg(\n        ranking_score=(\'ranking_score\', \'mean\'), valid_graphs=(\'file\', \'count\'))\n    table[\'collection\'] = collection\n    gnn_score = table.loc[table[\'model\'] == \'gnn\', \'ranking_score\'].iloc[0]\n    table[\'gnn_minus_model\'] = gnn_score - table[\'ranking_score\']\n    table[\'comparison_kind\'] = (\'saved baseline members\' if saved is not None\n                                else \'retrained references; not top-17% artifact\')\n    table.to_csv(run_dir / \'comparison_summary.csv\', index=False)\n    display(detail.pivot(index=\'file\', columns=\'model\', values=\'ranking_score\'))\n    display(table)\n    return table\n\n'
gnn = {"display": display, "npy_member_metadata": npy_member_metadata,
       "num_configs_in_npz": num_configs_in_npz, "read_npz_rows": read_npz_rows,
       "iter_npz_row_batches": iter_npz_row_batches,
       "load_npz_without_node_configs": load_npz_without_node_configs,
       "SAMPLED_CONFIG_CACHE": SAMPLED_CONFIG_CACHE}
exec(GNN_SOURCE, gnn)
gnn["GNN_OUTPUT_DIR"] = FULL_OUTPUT / "gnn_runs"
gnn["GNN_EPOCHS"] = 20
gnn["GNN_VALIDATE_EVERY"] = 1
gnn["GNN_PATIENCE"] = 5
gnn_models = {}
comparison_tables = []
for collection in gnn["GNN_COLLECTIONS"]:
    if collection == "layout:xla:default":
        train_files = gnn["select_files_for_split"](collection, "train", 24, RANDOM_SEED)
    else:
        train_files = gnn["split_files"](collection, "train")[:8]
    valid_files = gnn["split_files"](collection, "valid")[:5]
    if not train_files or not valid_files:
        raise ValueError(f"Missing training or validation files for {collection}")
    coverage = gnn["show_training_coverage"](collection, train_files)
    model = gnn["LayoutGNNRanker"]()
    manifest = gnn["make_validation_manifest"](model, valid_files)
    gnn["FIXED_VALIDATION"][collection] = manifest
    folder = gnn["GNN_OUTPUT_DIR"] / collection.replace(":", "_")
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "validation_manifest.json").write_text(json.dumps(manifest, indent=2))
    coverage.to_csv(folder / "training_coverage.csv")
    model.fit(collection, train_files, valid_files=valid_files)
    gnn_models[collection] = model
    comparison = gnn["compare_models"](model, collection, train_files, valid_files,
                                         saved=ensemble_members_by_collection)
    comparison["comparison_kind"] = "baseline rebuilt in this run; not lost original artifacts"
    comparison_tables.append(comparison)
    model.cache.clear(remove_disk=True)
    gc.collect()
    print("Cache after collection cleanup:", SAMPLED_CONFIG_CACHE.report())
validation_comparison = pd.concat(comparison_tables, ignore_index=True)
validation_comparison.to_csv(FULL_OUTPUT / "validation_comparison.csv", index=False)
display(validation_comparison)


## 7. Export the hybrid candidate

The hybrid swaps both XLA layout collections as requested. It does so even if
one GNN has a lower diagnostic score, and prints the comparison to make that
visible. The remaining three collections are copied byte-for-byte at the CSV
field level from this run's baseline. This does not claim that the lost historical
submission is reproduced. No Kaggle submission is sent automatically.


In [ ]:
TARGET_COLLECTIONS = {"layout:xla:default", "layout:xla:random"}

def validate_candidate(candidate, baseline, template, counts):
    if list(candidate.columns) != ["ID", "TopConfigs"] or candidate["ID"].duplicated().any():
        raise ValueError("Invalid submission columns or duplicate IDs")
    if candidate["ID"].tolist() != template["ID"].tolist():
        raise ValueError("Submission IDs/order differ from the template")
    if baseline["ID"].tolist() != candidate["ID"].tolist():
        raise ValueError("Baseline and hybrid ID order differ")
    expected = dict(zip(template["ID"], template["TopConfigs"].map(top_config_count)))
    for i, row in candidate.iterrows():
        collection, _ = id_to_collection_and_stem(row["ID"])
        values = [int(v) for v in str(row["TopConfigs"]).split(";")]
        if len(values) != expected[row["ID"]] or len(values) != len(set(values)):
            raise ValueError(f"Wrong length or duplicate configuration: {row['ID']}")
        if any(v < 0 or v >= counts[row["ID"]] for v in values):
            raise ValueError(f"Out-of-range configuration: {row['ID']}")
        if collection not in TARGET_COLLECTIONS and row["TopConfigs"] != baseline.iloc[i]["TopConfigs"]:
            raise ValueError(f"An untargeted collection changed: {collection}")
    return {"rows": len(candidate), "target_collections": sorted(TARGET_COLLECTIONS),
            "other_three_collections_unchanged": True}

template = pd.read_csv(SAMPLE_SUBMISSION_PATH)
hybrid_submission = baseline_submission.copy(deep=True)
config_counts = {}
for i, row in tqdm(template.iterrows(), total=len(template), desc="Hybrid prediction"):
    collection, stem = id_to_collection_and_stem(row["ID"])
    path = COLLECTIONS[collection] / "test" / f"{stem}.npz"
    expected_count = top_config_count(row["TopConfigs"])
    if collection in TARGET_COLLECTIONS:
        indices, scores = gnn_models[collection].predict_file(path, split="test", max_configs=None)
        if not np.isfinite(scores).all() or len(indices) < expected_count:
            raise ValueError(f"Invalid GNN predictions for {row['ID']}")
        ordered = indices[np.argsort(scores, kind="stable")][:expected_count]
        hybrid_submission.at[i, "TopConfigs"] = ";".join(map(str, ordered.tolist()))
        config_counts[row["ID"]] = len(indices)
    else:
        config_counts[row["ID"]] = num_configs_in_npz(path)

check = validate_candidate(hybrid_submission, baseline_submission, template, config_counts)
hybrid_submission.to_csv(FULL_OUTPUT / "submission_hybrid.csv", index=False)
(FULL_OUTPUT / "submission_checks.json").write_text(json.dumps(check, indent=2))
print(check)
for collection in sorted(TARGET_COLLECTIONS):
    table = validation_comparison[validation_comparison.collection == collection].set_index("model")
    delta = table.loc["gnn", "ranking_score"] - table.loc["saved_main_ensemble", "ranking_score"]
    print(f"{collection}: GNN minus rebuilt baseline = {delta:+.5f}")
    if delta <= 0:
        print("GNN did not beat the rebuilt baseline on this sample; the hybrid remains an experimental candidate.")
print("Baseline:", FULL_OUTPUT / "submission_baseline.csv")
print("Hybrid candidate:", FULL_OUTPUT / "submission_hybrid.csv")
print("Keep the baseline and validation report when comparing your Kaggle scores.")


## 8. Download results and model backups

In [ ]:
print("Final sampled-cache report:", SAMPLED_CONFIG_CACHE.report())
SAMPLED_CONFIG_CACHE.clear()
print("Cleared temporary sampled configuration cache before archiving.")
bundle = shutil.make_archive(str(FULL_OUTPUT.parent / "full_run_backup"), "zip", FULL_OUTPUT)
print("Backup bundle:", bundle)
try:
    from google.colab import files
    files.download(str(FULL_OUTPUT / "submission_hybrid.csv"))
    files.download(bundle)
except ImportError:
    print("Outside Colab: copy the CSV and backup bundle from the printed paths.")
